Dataset: FoggyCityscapes (first 100 images)

Pseudo-label generation:

CLIP-only pseudo-labels → saved to OUT/clip_only/{imgs,lbls,meta}

CLIP+BLIP pseudo-labels → saved to OUT/clip_plus_blip/{imgs,lbls,meta}

Caps:

final kept SAM masks per image ≤ 15

BLIP called only on CLIP-“unlabeled” masks, ≤ 5 per image

Model:

SAM (vit_h)

EVA-CLIP checkpoint: eva_clip_ft_cityscapes_ctxfusion_19cls.pt

BLIP caption model: Salesforce/blip-image-captioning-base on CPU

Evaluation:

runs your sam_clip_full.seg.seg_main eval on both pseudo datasets (adjust CLI args if your eval differs)

In [1]:
import os
import sys
import json
import shutil
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from tqdm import tqdm


In [2]:
REPO_ROOT = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis"
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

# ---- your modules ----
from sam_clip_full.sam_module import load_sam_predictor, sam_predict_masks
from sam_clip_full.clip_module import (
    load_eva_clip,
    EVACLIPWrapper,
    _mask_to_bbox,
    apply_soft_mask_weighting,
    dynamic_alpha_beta_from_box,
    crop_image,
    DEFAULT_TEMPLATES,
    load_finetuned_checkpoint,
)

# ---- BLIP ----
from transformers import BlipProcessor, BlipForConditionalGeneration


/home/scs_deal_projects_notapebackup/user/shubhang/thesis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
NUM_IMAGES = 100


device: cuda


In [4]:
# -----------------------------
# FoggyCityscapes images
# -----------------------------
from pathlib import Path

FOGGY_IMG_DIR = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/leftImg8bit_foggyDBF/val")
assert FOGGY_IMG_DIR.exists(), f"Missing: {FOGGY_IMG_DIR}"

# Cityscapes-style layout: val/<city>/*.png
foggy_paths = sorted(FOGGY_IMG_DIR.glob("*/*.png"))[:NUM_IMAGES]

print("Foggy images selected:", len(foggy_paths))
print("Example:", foggy_paths[0] if foggy_paths else None)
assert len(foggy_paths) > 0, "Still 0 images — check the folder structure under leftImg8bit_foggyDBF/val"


# -----------------------------
# Outputs
# -----------------------------
OUT_ROOT = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_foggy_cityscapes19_bliptest")
OUT_CLIP      = OUT_ROOT / "clip_only"
OUT_CLIP_BLIP = OUT_ROOT / "clip_plus_blip"

for d in [OUT_CLIP, OUT_CLIP_BLIP]:
    (d / "imgs").mkdir(parents=True, exist_ok=True)
    (d / "lbls").mkdir(parents=True, exist_ok=True)
    (d / "meta").mkdir(parents=True, exist_ok=True)

# -----------------------------
# Thresholds & caps
# -----------------------------
TAU_CLIP_PROB = 0.30   # CLIP classification acceptance threshold
TAU_MAP_SIM   = 0.20   # BLIP-caption->class similarity threshold (used inside accept_mapping)
MAX_MASKS_PER_IMAGE = 15
MAX_BLIP_PER_IMAGE  = 5

IGNORE_LABEL = 255  # Cityscapes ignore index


Foggy images selected: 100
Example: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/leftImg8bit_foggyDBF/val/frankfurt/frankfurt_000000_000294_leftImg8bit_foggy_beta_0.005.png


In [5]:
# Cityscapes-19 (trainId order 0..18)
CAT_NAMES = [
    "road",          # 0
    "sidewalk",      # 1
    "building",      # 2
    "wall",          # 3
    "fence",         # 4
    "pole",          # 5
    "traffic light", # 6
    "traffic sign",  # 7
    "vegetation",    # 8
    "terrain",       # 9
    "sky",           # 10
    "person",        # 11
    "rider",         # 12
    "car",           # 13
    "truck",         # 14
    "bus",           # 15
    "train",         # 16
    "motorcycle",    # 17
    "bicycle",       # 18
]
NUM_CLASSES = len(CAT_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)


NUM_CLASSES: 19


In [6]:
# ---- SAM predictor ----
sam_predictor = load_sam_predictor(
    device=device,
    backend="sam1",   # <--- IMPORTANT
    model_type="vit_h",
    checkpoint="/home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/sam_vit_h_4b8939.pth",
)
print("SAM1 loaded.")



SAM1 loaded.


In [7]:
# ---- EVA-CLIP base ----
clip_model, clip_preprocess, clip_tokenizer = load_eva_clip(
    device=device,
    model_name="EVA02-L-14",
    pretrained="merged2b_s4b_b131k",
    to_float32=False,
)

# ---- load your CITYSCAPES finetuned checkpoint ----
ckpt_path = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_cityscapes_ctxfusion_19cls.pt"
import torch

ckpt = torch.load(ckpt_path, map_location="cpu")

# find state dict
if isinstance(ckpt, dict) and "model" in ckpt:
    sd = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    sd = ckpt["state_dict"]
elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    sd = ckpt["model_state_dict"]
elif isinstance(ckpt, dict) and any(isinstance(v, torch.Tensor) for v in ckpt.values()):
    sd = ckpt  # raw state_dict stored as dict
else:
    raise ValueError(f"Unknown checkpoint format. Keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

missing = clip_model.load_state_dict(sd, strict=False)
print("Loaded. Missing/unexpected:", missing)

# optional: restore logit_scale if saved
if isinstance(ckpt, dict) and "logit_scale_exp" in ckpt:
    with torch.no_grad():
        clip_model.logit_scale.copy_(torch.log(torch.tensor(ckpt["logit_scale_exp"])))
    print("Restored logit_scale from logit_scale_exp.")

print("Loaded finetuned EVA-CLIP checkpoint from:", ckpt_path)

eva = EVACLIPWrapper(
    clip_model,
    clip_preprocess,
    clip_tokenizer,
    device=device,
    combine="add",        # matches your ctx-fusion setup
    embed_dim=1024,
).to(device).eval()
print("EVA wrapper ready.")


Loaded. Missing/unexpected: <All keys matched successfully>
Loaded finetuned EVA-CLIP checkpoint from: /home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_cityscapes_ctxfusion_19cls.pt
EVA wrapper ready.


In [8]:
# ---- build text embeddings for Cityscapes-19 ----
with torch.no_grad():
    text_embeds = eva.build_text_cache(
        labels=CAT_NAMES,
        templates=DEFAULT_TEMPLATES,
        layout="mean",
    ).to(device)
    text_mean = eva.compute_text_mean(text_embeds).to(device)

print("text_embeds:", tuple(text_embeds.shape), "text_mean:", tuple(text_mean.shape))


text_embeds: (19, 768) text_mean: (1, 768)


In [9]:
blip_device = "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(blip_device).eval()

print("BLIP loaded on:", blip_device)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


BLIP loaded on: cpu


In [10]:
def pil_from_path(p: Path) -> Image.Image:
    return Image.open(p).convert("RGB")

def sam_on_pil(pil_img: Image.Image):
    np_img = np.array(pil_img)  # RGB uint8
    masks_list = sam_predict_masks(sam_predictor, np_img)  # list of HxW uint8 {0,1}
    masks_bool = [m.astype(bool) for m in masks_list]
    return masks_bool

def _mask_area(mask_bool: np.ndarray) -> int:
    return int(mask_bool.sum())

def select_top_sam_masks(masks_bool, top_k=60, min_area=300):
    """
    Cheap early pruning to reduce SAM clutter.
    - keep only masks above min_area
    - then take top_k by area
    """
    kept = [m for m in masks_bool if _mask_area(m) >= min_area]
    if len(kept) == 0:
        return []
    areas = np.array([_mask_area(m) for m in kept], dtype=np.int64)
    order = np.argsort(-areas)
    keep = order[: min(top_k, len(order))]
    return [kept[i] for i in keep]

def _safe_stem(p: Path) -> str:
    return p.name.replace(".png", "")

def save_pseudo(out_dir: Path, img_path: Path, label_map: np.ndarray, meta: dict):
    stem = _safe_stem(img_path)

    out_img = out_dir / "imgs" / f"{stem}.png"
    if not out_img.exists():
        shutil.copy2(img_path, out_img)

    out_lbl = out_dir / "lbls" / f"{stem}.png"
    Image.fromarray(label_map).save(out_lbl)

    out_meta = out_dir / "meta" / f"{stem}.json"
    meta_clean = _json_sanitize(meta)
    with open(out_meta, "w") as f:
        json.dump(meta_clean, f, indent=2)



In [11]:
#fog remove

import numpy as np
from PIL import Image, ImageFilter

def _sobel_mag(gray01: np.ndarray) -> np.ndarray:
    # gray01: HxW float32 in [0,1]
    kx = np.array([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=np.float32)
    ky = np.array([[1,2,1],[0,0,0],[-1,-2,-1]], dtype=np.float32)

    # cheap conv (no scipy)
    H, W = gray01.shape
    pad = np.pad(gray01, ((1,1),(1,1)), mode="edge")
    gx = np.zeros((H,W), np.float32)
    gy = np.zeros((H,W), np.float32)
    for i in range(3):
        for j in range(3):
            gx += kx[i,j] * pad[i:i+H, j:j+W]
            gy += ky[i,j] * pad[i:i+H, j:j+W]
    return np.sqrt(gx*gx + gy*gy)

def crop_mask_stats(pil_crop: Image.Image, mask_crop_bool: np.ndarray | None = None):
    """
    Returns simple texture stats on the crop (optionally only inside mask):
      - std: contrast
      - edge_mean: average gradient magnitude
      - edge_density: fraction of pixels with strong edges
    """
    g = np.asarray(pil_crop.convert("L"), dtype=np.float32) / 255.0
    if mask_crop_bool is not None:
        mc = mask_crop_bool.astype(bool)
        if mc.sum() < 50:
            return {"std": 0.0, "edge_mean": 0.0, "edge_density": 0.0}
        vals = g[mc]
        std = float(vals.std())
        mag = _sobel_mag(g)
        magv = mag[mc]
    else:
        std = float(g.std())
        mag = _sobel_mag(g)
        magv = mag.reshape(-1)

    edge_mean = float(magv.mean())
    edge_density = float((magv > 0.08).mean())  # threshold tuned-ish; adjust if needed
    return {"std": std, "edge_mean": edge_mean, "edge_density": edge_density}

def is_fog_like_crop(pil_crop: Image.Image, mask_crop_bool: np.ndarray | None = None,
                     std_thr=0.07, edge_mean_thr=0.03, edge_density_thr=0.05):
    s = crop_mask_stats(pil_crop, mask_crop_bool)
    # fog-like = low contrast + few edges
    return (s["std"] < std_thr) and (s["edge_mean"] < edge_mean_thr) and (s["edge_density"] < edge_density_thr), s


In [12]:
import numpy as np

def bbox_area_frac(b, H, W):
    x0,y0,x1,y1 = b
    return ((x1-x0)*(y1-y0)) / float(H*W)

def compute_uncertainty_from_logits(logits_np, logit_scale_exp, temp_calib=0.07):
    # logits_np: (C,)
    cos = logits_np / max(logit_scale_exp, 1e-8)  # approximate cosine sims
    order = np.argsort(-cos)
    top1 = int(order[0])
    top2 = int(order[1]) if len(order) > 1 else int(order[0])

    margin_cos = float(cos[top1] - cos[top2])

    # calibrated distribution (softmax on cosine / temp)
    z = cos / max(temp_calib, 1e-8)
    z = z - z.max()
    p = np.exp(z); p = p / (p.sum() + 1e-12)

    pcal_top1 = float(p[top1])
    entropy = float(-(p * np.log(p + 1e-12)).sum())
    return top1, top2, margin_cos, pcal_top1, entropy

def should_trigger_blip(label1, label2, margin_cos, pcal_top1, entropy, bbox_frac):
    # conservative gates that worked for you
    if bbox_frac > 0.20:      # huge regions (road/sky) usually not helpful
        return False
    if bbox_frac < 0.01:      # tiny specks
        return False
    # uncertainty
    return (margin_cos <= 0.12) or (pcal_top1 <= 0.60) or (entropy >= 1.6)

def should_accept_blip(mapped_label, mapped_sim, clip_label1, clip_label2):
    # accept only if BLIP agrees with CLIP's top-2 hypothesis (prevents random captions)
    if mapped_label not in {clip_label1, clip_label2}:
        return False
    return float(mapped_sim) >= 0.55


In [13]:
def bbox_area_frac(b, H, W):
    x0,y0,x1,y1 = b
    return ((x1-x0)*(y1-y0)) / float(H*W + 1e-8)

def prune_sam_masks_with_bg(pil_img, masks_bool):
    W,H = pil_img.size
    keep = []
    hard_ignore = []
    skipped_small = 0
    skipped_bg = 0

    for m in masks_bool:
        m = m.astype(bool)
        area = int(m.sum())
        if area < MIN_MASK_AREA:
            skipped_small += 1
            continue

        bbox = _mask_to_bbox(m, pad=4)
        if bbox is None:
            continue

        area_frac = area / float(H*W)
        bb_frac   = bbox_area_frac(bbox, H, W)

        # Only run expensive fog test on *big* masks
        if (area_frac >= MAX_AREA_FRAC_BG) or (bb_frac >= MAX_BBOX_FRAC_BG):
            x0,y0,x1,y1 = bbox
            crop_raw = crop_image(pil_img, bbox)
            mask_crop_bool = m[y0:y1, x0:x1].astype(bool)

            fog_like, stats = is_fog_like_crop(
                crop_raw,
                mask_crop_bool=mask_crop_bool,
                std_thr=FOG_STD_THR,
                edge_mean_thr=FOG_EDGE_MEAN_THR,
                edge_density_thr=FOG_EDGE_DENS_THR,
            )
            if fog_like:
                hard_ignore.append(m)   # <- lock as IGNORE so nothing overwrites it
                skipped_bg += 1
                continue

        keep.append(m)

    # area sort + cap
    keep = sorted(keep, key=lambda x: int(x.sum()), reverse=True)[:MAX_SAM_MASKS]
    return keep, hard_ignore, {"skipped_small": skipped_small, "skipped_bg": skipped_bg}


def build_semantic_label_map(H, W, instances, ignore_label=IGNORE_LABEL, hard_ignore_masks=None):
    lbl = np.full((H, W), fill_value=ignore_label, dtype=np.uint8)
    score_map = np.full((H, W), fill_value=-1.0, dtype=np.float32)

    # Lock pixels that we consider "fog/background"
    if hard_ignore_masks:
        hard = np.zeros((H,W), dtype=bool)
        for m in hard_ignore_masks:
            hard |= m.astype(bool)
        score_map[hard] = 1e9           # nobody can beat this
        lbl[hard] = ignore_label

    for inst in instances:
        m = inst["mask"].astype(bool)
        cid = int(inst["class_id"])
        s = float(inst["score"])
        take = m & (s > score_map)
        lbl[take] = cid
        score_map[take] = s

    return lbl



In [14]:
def apply_soft_mask_weighting_strong(crop_pil, mask_crop, bg_weight=0.0):
    if mask_crop is None or mask_crop.size == 0:
        return crop_pil
    m = Image.fromarray((mask_crop.astype(np.uint8) * 255)).resize(crop_pil.size, Image.NEAREST)
    m = (np.array(m) > 127)
    img = np.array(crop_pil).astype(np.float32)
    w = np.where(m[...,None], 1.0, bg_weight)
    out = np.clip(img * w + 127.0*(1.0-w), 0, 255).astype(np.uint8)  # gray bg
    return Image.fromarray(out)


In [15]:
@torch.no_grad()
def classify_masks_with_clip(pil_img, masks_bool):
    W, H = pil_img.size
    results = []

    for midx, mask in enumerate(masks_bool):
        bbox = _mask_to_bbox(mask, pad=4)
        if bbox is None:
            continue
        x0,y0,x1,y1 = bbox

        crop = crop_image(pil_img, bbox)
        mask_crop = mask[y0:y1, x0:x1]
        crop_soft = apply_soft_mask_weighting_strong(crop, mask_crop)

        alpha, beta, _ = dynamic_alpha_beta_from_box((x0,y0,x1,y1), W, H)

        logits_by_class, probs, *_ = eva.classify_crop_with_context_residual_topk(
            crop_pil=crop_soft,
            context_pil=pil_img,
            text_embeds=text_embeds,
            text_feat_mean=text_mean,
            alpha=alpha, beta=beta,
            cat_names=CAT_NAMES,
            templates=DEFAULT_TEMPLATES,
            topk_list=(1,3,5),
        )

        logits_np = logits_by_class.detach().cpu().numpy()  # ✅ add this
        probs_np  = probs.detach().cpu().numpy()

        results.append({
            "mask_index": midx,
            "bbox": bbox,
            "mask": mask,
            "logits": logits_by_class.detach().cpu().numpy(),        # ✅
            "probs": probs_np,
            "max_prob": float(probs_np.max()),
            "pred_idx": int(probs_np.argmax()),
            "pred_label": CAT_NAMES[int(probs_np.argmax())],
        })
    return results


In [16]:
@torch.no_grad()
def blip_caption(
    pil_images: List[Image.Image],
    prompt: Optional[str] = None,
    batch_size: int = 4,
    max_new_tokens: int = 15,
) -> List[str]:
    captions_all: List[str] = []
    n = len(pil_images)
    if n == 0:
        return captions_all

    for start in range(0, n, batch_size):
        batch = pil_images[start:start + batch_size]

        if prompt is None:
            inputs = processor(images=batch, return_tensors="pt")
        else:
            inputs = processor(images=batch, text=[prompt] * len(batch), return_tensors="pt")

        inputs = {k: v.to(blip_device) for k, v in inputs.items()}

        out = blip_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
        )
        caps = processor.batch_decode(out, skip_special_tokens=True)
        captions_all.extend([c.strip() for c in caps])

    return captions_all


In [17]:
# # Move EVA to CPU for caption mapping (text-only), keep class text_embeds on CPU as well
# eva_text_device = "cpu"
# eva.model = eva.model.to(eva_text_device)
# text_embeds_cpu = text_embeds.detach().cpu()

torch.cuda.empty_cache()
print("EVA text tower moved to CPU for caption mapping.")


EVA text tower moved to CPU for caption mapping.


In [18]:
@torch.no_grad()
def map_caption_to_class_gpu(caption: str) -> Dict[str, Any]:
    dev = eva.device
    toks = eva.tokenizer([caption]).to(dev)
    z_text = eva.model.encode_text(toks)
    z_text = F.normalize(z_text.float(), dim=-1)

    sims = (z_text @ text_embeds.T)[0]
    best_idx = int(torch.argmax(sims).item())
    best_sim = float(sims[best_idx].item())
    best_label = CAT_NAMES[best_idx]
    return {"best_idx": best_idx, "best_label": best_label, "best_sim": best_sim}


In [19]:
import re

ONTOLOGY_SYNONYMS = {
    "road": ["road", "street", "asphalt", "pavement"],
    "sidewalk": ["sidewalk", "footpath", "walkway", "pavement"],
    "building": ["building", "house", "apartment", "facade"],
    "wall": ["wall", "brick wall", "concrete wall"],
    "fence": ["fence", "railing", "barrier", "gate"],
    "pole": ["pole", "post", "lamppost", "streetlight"],
    "traffic light": ["traffic light", "signal", "stoplight"],
    "traffic sign": ["traffic sign", "street sign", "sign"],
    "vegetation": ["tree", "trees", "vegetation", "bush", "grass", "plants"],
    "terrain": ["terrain", "ground", "dirt", "soil", "gravel"],
    "sky": ["sky", "clouds"],
    "person": ["person", "man", "woman", "boy", "girl", "pedestrian"],
    "rider": ["rider", "cyclist", "motorcyclist", "person riding"],
    "car": ["car", "vehicle", "sedan", "automobile"],
    "truck": ["truck", "lorry"],
    "bus": ["bus", "coach"],
    "train": ["train", "tram"],
    "motorcycle": ["motorcycle", "motorbike", "bike"],
    "bicycle": ["bicycle", "bike", "cycle"],
}

def caption_contains_label_or_synonym(caption: str, label: str) -> bool:
    caption_l = caption.lower()
    words = set(re.findall(r"\w+", caption_l))
    syns = ONTOLOGY_SYNONYMS.get(label, [label])
    for s in syns:
        s = s.lower()
        if s in words or s in caption_l:
            return True
    return False

def accept_mapping(caption: str, mapping: Dict[str, Any]) -> bool:
    sim = float(mapping["best_sim"])
    label = mapping["best_label"]
    # conservative rule; tune later
    if sim >= 0.55:
        return True
    if sim >= 0.50 and caption_contains_label_or_synonym(caption, label):
        return True
    if sim >= TAU_MAP_SIM and caption_contains_label_or_synonym(caption, label):
        return True
    return False


In [20]:
def build_semantic_label_map(H: int, W: int, instances: list, ignore_label: int = IGNORE_LABEL):
    lbl = np.full((H, W), fill_value=ignore_label, dtype=np.uint8)
    score_map = np.full((H, W), fill_value=-1.0, dtype=np.float32)

    for inst in instances:
        m = inst["mask"]
        cid = int(inst["class_id"])
        s = float(inst["score"])
        take = m & (s > score_map)
        lbl[take] = cid
        score_map[take] = s

    return lbl


In [21]:
import numpy as np

def _json_sanitize(x):
    """
    Convert numpy / torch / Path types into JSON-serializable python types.
    """
    # numpy scalars
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    # numpy arrays (we don't want huge arrays in meta; handle small ones)
    if isinstance(x, np.ndarray):
        return x.tolist()
    # torch scalars/tensors
    try:
        import torch
        if isinstance(x, torch.Tensor):
            if x.numel() == 1:
                return x.item()
            return x.detach().cpu().tolist()
    except Exception:
        pass
    # pathlib
    if isinstance(x, Path):
        return str(x)
    # dict / list recursion
    if isinstance(x, dict):
        return {str(k): _json_sanitize(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_json_sanitize(v) for v in x]
    # default: return as-is (json will handle str/int/float/bool/None)
    return x


In [22]:
def force_eva_to_device(eva: EVACLIPWrapper, device: str):
    # move model
    eva.model = eva.model.to(device)

    # IMPORTANT: sync wrapper device to where the model actually is
    eva.device = next(eva.model.parameters()).device  # torch.device('cuda:0') or 'cpu'

    # sanity
    p = next(eva.model.parameters())
    print("EVA model param device:", p.device, "dtype:", p.dtype)
    print("EVA wrapper device:", eva.device)


In [23]:
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


torch.cuda.is_available(): True
torch.cuda.device_count(): 1
GPU name: NVIDIA RTX A4500


In [24]:
# Force wrapper + underlying model to GPU (if possible)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

eva.device = device
eva.model = eva.model.to(device)

# Verify
p = next(eva.model.parameters())
print("EVA param device:", p.device, "dtype:", p.dtype)



Using device: cuda
EVA param device: cuda:0 dtype: torch.float32


In [25]:
print("eva.device:", eva.device)
print("eva param device:", next(eva.model.parameters()).device)
print("text_embeds device:", text_embeds.device)


eva.device: cuda
eva param device: cuda:0
text_embeds device: cuda:0


In [26]:

# CLIP keep (strict -> makes BLIP get more chances)
CLIP_KEEP_PCAL   = 0.85
CLIP_KEEP_MARGIN = 0.18

# BLIP override routing (CLIP-kept but still ambiguous)
BLIP_OVERRIDE_MAX_PER_IMAGE = 3
BLIP_OVERRIDE_SIM_MIN       = 0.66   # require stronger BLIP to override
BLIP_OVERRIDE_BONUS         = 0.20   # extra bonus ONLY when overriding CLIP

# BLIP general score boost (helps it win overlaps vs CLIP sometimes)
BLIP_SCORE_BOOST            = 0.10   # mild. don't go crazy.
BLIP_MIN_SIM_ACCEPT         = 0.62   # for normal (non-override) accept



In [27]:
import numpy as np
from PIL import Image, ImageFilter

def make_blip_focus_crop(
    pil_img: Image.Image,
    bbox_xyxy,
    mask_full_bool: np.ndarray,
    bg_mode: str = "gray",      # "gray" or "blur"
    blur_bg: bool = True,
    blur_radius: float = 6.0,
):
    """
    Crop bbox, but 'focus' the masked region:
      - outside-mask pixels become gray and/or blurred
    This helps BLIP not hallucinate based on background clutter.
    """
    x0, y0, x1, y1 = map(int, bbox_xyxy)
    crop = pil_img.crop((x0, y0, x1, y1)).convert("RGB")

    m = mask_full_bool[y0:y1, x0:x1].astype(bool)
    if m.size == 0 or m.sum() == 0:
        return crop  # fallback

    crop_np = np.array(crop)

    # background image
    if blur_bg:
        bg = crop.filter(ImageFilter.GaussianBlur(radius=blur_radius))
        bg_np = np.array(bg)
    else:
        bg_np = crop_np.copy()

    if bg_mode == "gray":
        gray = np.full_like(crop_np, 128, dtype=np.uint8)
        bg_np = gray if not blur_bg else (0.5 * bg_np + 0.5 * gray).astype(np.uint8)

    # composite: keep foreground pixels, replace background pixels
    out = crop_np.copy()
    out[~m] = bg_np[~m]
    return Image.fromarray(out)


In [28]:
def bbox_area_frac(b, H, W):
    x0,y0,x1,y1 = b
    return ((x1-x0)*(y1-y0)) / float(H*W + 1e-8)

def prune_sam_masks_with_bg(pil_img, masks_bool):
    W,H = pil_img.size
    keep = []
    hard_ignore = []
    skipped_small = 0
    skipped_bg = 0

    for m in masks_bool:
        m = m.astype(bool)
        area = int(m.sum())
        if area < MIN_MASK_AREA:
            skipped_small += 1
            continue

        bbox = _mask_to_bbox(m, pad=4)
        if bbox is None:
            continue

        area_frac = area / float(H*W)
        bb_frac   = bbox_area_frac(bbox, H, W)

        # Only run expensive fog test on *big* masks
        if (area_frac >= MAX_AREA_FRAC_BG) or (bb_frac >= MAX_BBOX_FRAC_BG):
            x0,y0,x1,y1 = bbox
            crop_raw = crop_image(pil_img, bbox)
            mask_crop_bool = m[y0:y1, x0:x1].astype(bool)

            fog_like, stats = is_fog_like_crop(
                crop_raw,
                mask_crop_bool=mask_crop_bool,
                std_thr=FOG_STD_THR,
                edge_mean_thr=FOG_EDGE_MEAN_THR,
                edge_density_thr=FOG_EDGE_DENS_THR,
            )
            if fog_like:
                hard_ignore.append(m)   # <- lock as IGNORE so nothing overwrites it
                skipped_bg += 1
                continue

        keep.append(m)

    # area sort + cap
    keep = sorted(keep, key=lambda x: int(x.sum()), reverse=True)[:MAX_SAM_MASKS]
    return keep, hard_ignore, {"skipped_small": skipped_small, "skipped_bg": skipped_bg}


def build_semantic_label_map(H, W, instances, ignore_label=IGNORE_LABEL, hard_ignore_masks=None):
    lbl = np.full((H, W), fill_value=ignore_label, dtype=np.uint8)
    score_map = np.full((H, W), fill_value=-1.0, dtype=np.float32)

    # Lock pixels that we consider "fog/background"
    if hard_ignore_masks:
        hard = np.zeros((H,W), dtype=bool)
        for m in hard_ignore_masks:
            hard |= m.astype(bool)
        score_map[hard] = 1e9           # nobody can beat this
        lbl[hard] = ignore_label

    for inst in instances:
        m = inst["mask"].astype(bool)
        cid = int(inst["class_id"])
        s = float(inst["score"])
        take = m & (s > score_map)
        lbl[take] = cid
        score_map[take] = s

    return lbl



In [29]:
_EVA_ON_DEVICE = None

def ensure_eva_image_on(device: str, verbose: bool = False):
    global eva, text_embeds, text_mean, _EVA_ON_DEVICE

    # already on that device -> do nothing
    if _EVA_ON_DEVICE == device:
        return

    eva.device = device
    eva.model = eva.model.to(device)
    text_embeds = text_embeds.to(device)
    text_mean   = text_mean.to(device)

    _EVA_ON_DEVICE = device

    if verbose:
        p = next(eva.model.parameters())
        print("[EVA] model device:", p.device, "| dtype:", p.dtype)
        print("[EVA] text_embeds:", text_embeds.device, "| text_mean:", text_mean.device)


In [30]:
#check if clip and blip are different

from pathlib import Path
import numpy as np
from PIL import Image

CLIP_LBLS = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_foggy_cityscapes19_bliptest/clip_only/lbls")
BLIP_LBLS = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_foggy_cityscapes19_bliptest/clip_plus_blip/lbls")
IGNORE=255

files = sorted(set(p.name for p in CLIP_LBLS.glob("*.png")) & set(p.name for p in BLIP_LBLS.glob("*.png")))
print("common files:", len(files))

tot = 0
diff = 0
for fn in files:
    a = np.array(Image.open(CLIP_LBLS/fn), dtype=np.int64)
    b = np.array(Image.open(BLIP_LBLS/fn), dtype=np.int64)

    # compare only where at least one labels something
    mask = (a != IGNORE) | (b != IGNORE)
    tot += int(mask.sum())
    diff += int((a[mask] != b[mask]).sum())

print("diff fraction on labeled pixels:", diff / max(tot,1))


common files: 100
diff fraction on labeled pixels: 0.052465607121546166


In [31]:
#gt labels

from pathlib import Path

FOGGY_IMG_DIR = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/leftImg8bit_foggyDBF/val")
GT_LBLS_DIR   = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/seg_data_foggy_trainId/val_lbls")

gt_names = sorted([p.name for p in GT_LBLS_DIR.glob("*.png")])
print("GT labels:", len(gt_names))

# map all foggy images by filename
foggy_map = {p.name: p for p in FOGGY_IMG_DIR.rglob("*.png")}

foggy_paths_gt = []
missing = []
for n in gt_names:
    p = foggy_map.get(n, None)
    if p is None:
        missing.append(n)
    else:
        foggy_paths_gt.append(p)

print("Matched foggy images:", len(foggy_paths_gt))
print("Missing:", len(missing))
print("Example matched:", foggy_paths_gt[0] if foggy_paths_gt else None)
if missing:
    print("Example missing:", missing[:5])


GT labels: 52
Matched foggy images: 52
Missing: 0
Example matched: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/leftImg8bit_foggyDBF/val/lindau/lindau_000000_000019_leftImg8bit_foggy_beta_0.01.png


In [30]:
@torch.no_grad()
def make_pseudo_clip_only_v2(pil_img: Image.Image, temp_calib=0.07):
    ensure_eva_image_on(device)

    W,H = pil_img.size
    masks_bool = sam_on_pil(pil_img)

    masks_keep, hard_ignore, info = prune_sam_masks_with_bg(pil_img, masks_bool)

    clip_results = classify_masks_with_clip(pil_img, masks_keep)
    logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

    kept = []
    for r in clip_results:
        top1, top2, margin_cos, pcal_top1, entropy = compute_uncertainty_from_logits(
            r["logits"], logit_scale_exp=logit_scale_exp, temp_calib=temp_calib
        )
        if (pcal_top1 >= CLIP_KEEP_PCAL) and (margin_cos >= CLIP_KEEP_MARGIN):
            kept.append({"mask": r["mask"], "class_id": top1, "score": pcal_top1, "bbox": r["bbox"]})


    for r in clip_results[:5]:  # just inspect first 5
        logits = r["logits"]
    # make sure it's numpy
        if torch.is_tensor(logits):
            logits = logits.detach().cpu().numpy()

        mn = float(logits.min())
        mx = float(logits.max())
        # print("scaled logits range:", mn, mx)

        cos = logits / max(logit_scale_exp, 1e-8)
        # print("cosine-ish range:", float(cos.min()), float(cos.max()))

    kept = sorted(kept, key=lambda x: x["score"], reverse=True)[:MAX_MASKS_PER_IMAGE]
    lbl = build_semantic_label_map(H, W, kept, ignore_label=IGNORE_LABEL, hard_ignore_masks=hard_ignore)

    meta = {
        "mode":"clip_only_v2",
        "n_sam_raw": len(masks_bool),
        "n_sam_keep": len(masks_keep),
        "n_hard_ignore": len(hard_ignore),
        **info,
        "n_kept": len(kept),
        "temp_calib": temp_calib,
        "CLIP_KEEP_PCAL": CLIP_KEEP_PCAL,
        "CLIP_KEEP_MARGIN": CLIP_KEEP_MARGIN,
    }
    return {"label": lbl, "meta": meta}


In [45]:
IGNORE_LABEL = 255

MAX_SAM_MASKS = 25
MIN_MASK_AREA = 800

# Big mask = likely haze/background blob
MAX_AREA_FRAC_BG = 0.55     # mask area / image area
MAX_BBOX_FRAC_BG = 0.80     # bbox area / image area

# Fog-like thresholds (tune lightly)
FOG_STD_THR         = 0.07
FOG_EDGE_MEAN_THR   = 0.03
FOG_EDGE_DENS_THR   = 0.05


In [32]:
# ---------------------------
# Knobs (keep these GLOBAL so both clip_only_v2 and clip_plus_blip_v2 share them)
# ---------------------------

# shared (must match clip_only_v2)
MAX_MASKS_PER_IMAGE  = 15
CLIP_KEEP_PCAL       = 0.85
CLIP_KEEP_MARGIN     = 0.18
IGNORE_LABEL         = 255

# BLIP routing (how much BLIP gets to try)
MAX_BLIP_PER_IMAGE          = 6     # rescue (from non-confident CLIP)
BLIP_OVERRIDE_MAX_PER_IMAGE = 3     # override (from CLIP-kept but ambiguous)

# BLIP accept / scoring
BLIP_MIN_SIM_ACCEPT   = 0.62        # rescue accept threshold
BLIP_SCORE_BOOST      = 0.10        # small boost so BLIP can win overlaps sometimes

# BLIP override behavior
BLIP_OVERRIDE_SIM_MIN = 0.66        # require stronger BLIP to override
BLIP_OVERRIDE_BONUS   = 0.20        # extra bonus only when overriding CLIP
BLIP_OVERRIDE_REQUIRE_TOP2 = False  # if True: only override if BLIP == CLIP top2[1]

# "ambiguous CLIP-kept" definition (controls how often override is attempted)
CLIP_OVERRIDE_MARGIN_SLACK = 0.06   # route to override if margin < (KEEP_MARGIN + slack)
CLIP_OVERRIDE_ENTROPY_MIN  = 1.60   # or if entropy is high


def _bbox_area_frac(bbox, H, W):
    x0,y0,x1,y1 = bbox
    return float(max(0, x1-x0) * max(0, y1-y0) / max(1, H*W))


@torch.no_grad()
def make_pseudo_clip_plus_blip_v2(pil_img: Image.Image, temp_calib=0.07):
    """
    v2: matches your clip_only_v2 pipeline:
      - SAM -> prune_sam_masks_with_bg -> masks_keep + hard_ignore + info
      - CLIP strict keep using (CLIP_KEEP_PCAL, CLIP_KEEP_MARGIN)
      - BLIP rescues non-confident masks (MAX_BLIP_PER_IMAGE)
      - BLIP can override some CLIP-kept-but-ambiguous masks (BLIP_OVERRIDE_MAX_PER_IMAGE)
      - build_semantic_label_map(..., hard_ignore_masks=hard_ignore)
    """
    ensure_eva_image_on(device)

    W, H = pil_img.size

    # 1) SAM + your background/fog pruning
    masks_bool = sam_on_pil(pil_img)
    masks_keep, hard_ignore, info = prune_sam_masks_with_bg(pil_img, masks_bool)

    # 2) CLIP logits for kept masks
    clip_results = classify_masks_with_clip(pil_img, masks_keep)
    logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

    clip_instances_by_mask = {}      # mask_index -> instance (kept by CLIP)
    blip_rescue_candidates = []      # from non-confident CLIP
    blip_override_candidates = []    # from CLIP-kept but ambiguous

    # helper: build a BLIP crop robustly
    def _make_blip_crop(bbox, m_bool):
        x0,y0,x1,y1 = bbox
        mask_crop_bool = m_bool[y0:y1, x0:x1].astype(bool)

        # preferred: your focused crop if it exists
        if "make_blip_focus_crop" in globals():
            try:
                return make_blip_focus_crop(pil_img, bbox, m_bool, bg_mode="gray", blur_bg=True)
            except Exception:
                pass

        # fallback: soft-mask crop
        crop = crop_image(pil_img, bbox)
        return apply_soft_mask_weighting(crop, mask_crop_bool)

    # 3) Decide CLIP-kept vs BLIP routing
    for r in clip_results:
        midx = int(r.get("mask_index", -1))
        bbox = r.get("bbox", None)
        if bbox is None:
            continue

        x0,y0,x1,y1 = bbox
        m = r["mask"].astype(bool)
        logits = r["logits"]  # IMPORTANT: classify_masks_with_clip must include this

        top1, top2, margin_cos, pcal_top1, entropy = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=temp_calib
        )

        label1 = CAT_NAMES[int(top1)]
        label2 = CAT_NAMES[int(top2)]
        frac = _bbox_area_frac(bbox, H, W)

        confident = (pcal_top1 >= CLIP_KEEP_PCAL) and (margin_cos >= CLIP_KEEP_MARGIN)

        if confident:
            # keep CLIP
            clip_instances_by_mask[midx] = {
                "mask_index": midx,
                "mask": m,
                "class_id": int(top1),
                "score": float(pcal_top1),
                "bbox": bbox,
                "src": "clip",
                "pcal": float(pcal_top1),
                "margin": float(margin_cos),
                "entropy": float(entropy),
                "top2": (label1, label2),
            }

            # route some "kept but ambiguous" to BLIP for override attempt
            ambiguous_kept = (margin_cos < (CLIP_KEEP_MARGIN + CLIP_OVERRIDE_MARGIN_SLACK)) or (entropy > CLIP_OVERRIDE_ENTROPY_MIN)
            if ambiguous_kept and (len(blip_override_candidates) < BLIP_OVERRIDE_MAX_PER_IMAGE * 6):
                blip_override_candidates.append({
                    "mask_index": midx,
                    "mask": m,
                    "bbox": bbox,
                    "crop_blip": _make_blip_crop(bbox, m),
                    "clip_label": label1,
                    "clip_top2": (label1, label2),
                    "margin_cos": float(margin_cos),
                    "pcal_top1": float(pcal_top1),
                    "entropy": float(entropy),
                    "bbox_frac": float(frac),
                    "mode": "override",
                })

        else:
            # route to BLIP rescue if your gating says yes
            if should_trigger_blip(label1, label2, margin_cos, pcal_top1, entropy, frac):
                blip_rescue_candidates.append({
                    "mask_index": midx,
                    "mask": m,
                    "bbox": bbox,
                    "crop_blip": _make_blip_crop(bbox, m),
                    "clip_top2": (label1, label2),
                    "margin_cos": float(margin_cos),
                    "pcal_top1": float(pcal_top1),
                    "entropy": float(entropy),
                    "bbox_frac": float(frac),
                    "mode": "rescue",
                })

    # 4) Limit BLIP work (pick most uncertain first)
    blip_rescue_candidates = sorted(
        blip_rescue_candidates, key=lambda e: (e["margin_cos"], -e["entropy"])
    )[:MAX_BLIP_PER_IMAGE]

    blip_override_candidates = sorted(
        blip_override_candidates, key=lambda e: (e["margin_cos"], -e["entropy"])
    )[:BLIP_OVERRIDE_MAX_PER_IMAGE]

    blip_candidates = blip_rescue_candidates + blip_override_candidates

    # 5) Run BLIP captions
    crops = [e["crop_blip"] for e in blip_candidates]
    captions = blip_caption(crops, batch_size=4, max_new_tokens=15) if crops else []

    blip_instances = []
    blip_rejected = []
    overrides_done = 0
    rescue_done = 0

    for e, cap in zip(blip_candidates, captions):
        mapping = map_caption_to_class_gpu(cap)
        mapped_idx   = int(mapping["best_idx"])
        mapped_label = mapping["best_label"]
        mapped_sim   = float(mapping["best_sim"])

        if e["mode"] == "rescue":
            ok = (
                (mapped_sim >= BLIP_MIN_SIM_ACCEPT) and
                should_accept_blip(mapped_label, mapped_sim, e["clip_top2"][0], e["clip_top2"][1])
            )

            if ok:
                score = min(1.0, mapped_sim + BLIP_SCORE_BOOST)
                blip_instances.append({
                    "mask_index": e["mask_index"],
                    "mask": e["mask"],
                    "class_id": mapped_idx,
                    "score": score,
                    "bbox": e["bbox"],
                    "src": "blip_rescue",
                    "caption": cap,
                    "mapped_label": mapped_label,
                    "mapped_sim": mapped_sim,
                    "clip_top2": e["clip_top2"],
                })
                rescue_done += 1
            else:
                blip_rejected.append({
                    "mode": "rescue",
                    "caption": cap,
                    "mapped": mapped_label,
                    "sim": mapped_sim,
                    "clip_top2": e["clip_top2"],
                })

        else:
            # override mode: BLIP can replace CLIP for SAME mask_index
            clip_label = e["clip_label"]
            clip_top2  = e["clip_top2"]

            override_ok = (mapped_sim >= BLIP_OVERRIDE_SIM_MIN) and (mapped_label != clip_label)

            if BLIP_OVERRIDE_REQUIRE_TOP2:
                override_ok = override_ok and (mapped_label == clip_top2[1])

            if override_ok:
                # remove CLIP instance for this mask_index
                clip_instances_by_mask.pop(e["mask_index"], None)

                score = min(1.0, mapped_sim + BLIP_SCORE_BOOST + BLIP_OVERRIDE_BONUS)
                blip_instances.append({
                    "mask_index": e["mask_index"],
                    "mask": e["mask"],
                    "class_id": mapped_idx,
                    "score": score,
                    "bbox": e["bbox"],
                    "src": "blip_override",
                    "caption": cap,
                    "mapped_label": mapped_label,
                    "mapped_sim": mapped_sim,
                    "clip_prev": clip_label,
                    "clip_top2": clip_top2,
                })
                overrides_done += 1
            else:
                blip_rejected.append({
                    "mode": "override",
                    "caption": cap,
                    "mapped": mapped_label,
                    "sim": mapped_sim,
                    "clip_label": clip_label,
                    "clip_top2": clip_top2,
                })

    # 6) Merge + cap final masks
    clip_instances = list(clip_instances_by_mask.values())
    merged = clip_instances + blip_instances
    merged = sorted(merged, key=lambda x: x["score"], reverse=True)[:MAX_MASKS_PER_IMAGE]

    # 7) Build label map (IMPORTANT: include hard_ignore masks)
    lbl = build_semantic_label_map(
        H, W, merged,
        ignore_label=IGNORE_LABEL,
        hard_ignore_masks=hard_ignore
    )

    meta = {
        "mode": "clip_plus_blip_v2",
        "n_sam_raw": len(masks_bool),
        "n_sam_keep": len(masks_keep),
        "n_hard_ignore": len(hard_ignore),
        **info,

        "n_clip_results": len(clip_results),
        "n_clip_kept": len(clip_instances),

        "n_blip_rescue_candidates": len(blip_rescue_candidates),
        "n_blip_override_candidates": len(blip_override_candidates),
        "n_blip_candidates": len(blip_candidates),

        "n_blip_accepted_total": len(blip_instances),
        "n_blip_rescues_done": rescue_done,
        "n_overrides_done": overrides_done,

        "n_final_merged": len(merged),

        "temp_calib": temp_calib,

        # snapshot knobs for reproducibility
        "MAX_MASKS_PER_IMAGE": MAX_MASKS_PER_IMAGE,
        "CLIP_KEEP_PCAL": CLIP_KEEP_PCAL,
        "CLIP_KEEP_MARGIN": CLIP_KEEP_MARGIN,
        "MAX_BLIP_PER_IMAGE": MAX_BLIP_PER_IMAGE,
        "BLIP_OVERRIDE_MAX_PER_IMAGE": BLIP_OVERRIDE_MAX_PER_IMAGE,
        "BLIP_MIN_SIM_ACCEPT": BLIP_MIN_SIM_ACCEPT,
        "BLIP_SCORE_BOOST": BLIP_SCORE_BOOST,
        "BLIP_OVERRIDE_SIM_MIN": BLIP_OVERRIDE_SIM_MIN,
        "BLIP_OVERRIDE_BONUS": BLIP_OVERRIDE_BONUS,
        "BLIP_OVERRIDE_REQUIRE_TOP2": BLIP_OVERRIDE_REQUIRE_TOP2,
        "CLIP_OVERRIDE_MARGIN_SLACK": CLIP_OVERRIDE_MARGIN_SLACK,
        "CLIP_OVERRIDE_ENTROPY_MIN": CLIP_OVERRIDE_ENTROPY_MIN,

        # small preview for debugging (keep tiny!)
        "blip_rejected_preview": blip_rejected[:5],
    }

    return {"label": lbl, "meta": meta}


In [43]:
from pathlib import Path
from tqdm import tqdm

OUT_ROOT_GT = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_foggy_cityscapes19_bliptest_GTVAL")
OUT_CLIP_GT = OUT_ROOT_GT / "clip_only_v2"
OUT_CLIPBLIP_GT = OUT_ROOT_GT / "clip_plus_blip_v2"

for d in [OUT_CLIP_GT, OUT_CLIPBLIP_GT]:
    (d / "imgs").mkdir(parents=True, exist_ok=True)
    (d / "lbls").mkdir(parents=True, exist_ok=True)
    (d / "meta").mkdir(parents=True, exist_ok=True)

# run on only GT-matched images
ensure_eva_image_on(device)

for img_path in tqdm(foggy_paths_gt, desc="CLIP-only v2 (GT set)"):
    out = make_pseudo_clip_only_v2(pil_from_path(img_path), temp_calib=0.07)
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIP_GT, img_path, out["label"], meta)

for img_path in tqdm(foggy_paths_gt, desc="CLIP+BLIP v2 (GT set)"):
    out = make_pseudo_clip_plus_blip_v2(pil_from_path(img_path), temp_calib=0.07)
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIPBLIP_GT, img_path, out["label"], meta)

print("Saved to:", OUT_ROOT_GT)


CLIP-only v2 (GT set):   0%|          | 0/52 [00:00<?, ?it/s]

CLIP+BLIP v2 (GT set): 100%|██████████| 52/52 [06:05<00:00,  7.03s/it]

Saved to: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_foggy_cityscapes19_bliptest_GTVAL


In [39]:
from pathlib import Path
import numpy as np
from PIL import Image

IGNORE = 255
NUM_CLASSES = 19

def fast_confmat(gt, pr, k):
    return np.bincount(k*gt + pr, minlength=k*k).reshape(k, k)

def eval_pseudo_vs_gt_common(pseudo_lbl_dir, gt_lbl_dir):
    pseudo_lbl_dir = Path(pseudo_lbl_dir)
    gt_lbl_dir = Path(gt_lbl_dir)

    gt_files = {p.name: p for p in gt_lbl_dir.glob("*.png")}
    pr_files = {p.name: p for p in pseudo_lbl_dir.glob("*.png")}
    common = sorted(set(gt_files.keys()) & set(pr_files.keys()))
    print("common files:", len(common))

    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    valid_gt_total = labeled_on_valid_gt = correct_on_overlap = overlap_total = 0

    for fn in common:
        gt = np.array(Image.open(gt_files[fn]), dtype=np.int64)
        pr = np.array(Image.open(pr_files[fn]), dtype=np.int64)

        valid_gt = (gt != IGNORE)
        labeled  = (pr != IGNORE)
        overlap  = valid_gt & labeled

        valid_gt_total     += int(valid_gt.sum())
        labeled_on_valid_gt += int(overlap.sum())

        if overlap.any():
            correct_on_overlap += int((pr[overlap] == gt[overlap]).sum())
            overlap_total      += int(overlap.sum())
            cm += fast_confmat(gt[overlap].astype(np.int64), pr[overlap].astype(np.int64), NUM_CLASSES)

    coverage = labeled_on_valid_gt / max(valid_gt_total, 1)
    pixacc   = correct_on_overlap / max(overlap_total, 1)

    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    denom = tp + fp + fn
    iou = np.where(denom > 0, tp / denom, np.nan)

    present = np.where(cm.sum(1) > 0)[0]
    miou_present = float(np.nanmean(iou[present])) if len(present) else float("nan")

    return {
        "images_used": len(common),
        "coverage_on_valid_gt": float(coverage),
        "pixel_acc_on_labeled_pixels": float(pixacc),
        "mIoU_on_labeled_pixels_present_classes": miou_present,
    }

In [ ]:
margins = []
pcals = []
logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

for img_path in foggy_paths[:5]:
    pil_img = pil_from_path(img_path)
    masks = select_top_sam_masks(sam_on_pil(pil_img), top_k=25, min_area=800)
    clip_results = classify_masks_with_clip(pil_img, masks)

    for r in clip_results:
        logits = r["logits"]
        if torch.is_tensor(logits):
            logits = logits.detach().cpu().numpy()

        top1, top2, margin_cos, pcal_top1, ent = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=0.07
        )
        margins.append(margin_cos)
        pcals.append(pcal_top1)

print("margin_cos: min/med/max", np.min(margins), np.median(margins), np.max(margins))
print("pcal_top1:  min/med/max", np.min(pcals), np.median(pcals), np.max(pcals))


margin_cos: min/med/max 0.0022204741835594177 0.19632917642593384 0.38962146639823914
pcal_top1:  min/med/max 0.20192629098892212 0.8578221797943115 0.9815785884857178


In [46]:
GT_LBLS = "/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/seg_data_foggy_trainId/val_lbls"
CLIP_LBLS = str(OUT_CLIP_GT / "lbls")
CLIP_BLIP_LBLS = str(OUT_CLIPBLIP_GT / "lbls")

print("[PSEUDO vs GT] CLIP-only:", eval_pseudo_vs_gt_common(CLIP_LBLS, GT_LBLS))
print("[PSEUDO vs GT] CLIP+BLIP:", eval_pseudo_vs_gt_common(CLIP_BLIP_LBLS, GT_LBLS))


common files: 52


/local/temp_shubhang/ipykernel_3355393/674545543.py:46: RuntimeWarning: invalid value encountered in divide
  iou = np.where(denom > 0, tp / denom, np.nan)


[PSEUDO vs GT] CLIP-only: {'images_used': 52, 'coverage_on_valid_gt': 0.4201217171291945, 'pixel_acc_on_labeled_pixels': 0.0020749168177837084, 'mIoU_on_labeled_pixels_present_classes': 0.0006938699480487149}
common files: 52
[PSEUDO vs GT] CLIP+BLIP: {'images_used': 52, 'coverage_on_valid_gt': 0.4258412640738637, 'pixel_acc_on_labeled_pixels': 0.11269183459969646, 'mIoU_on_labeled_pixels_present_classes': 0.03165885678959623}


## cityscapes-gta

In [34]:
# ---- EVA-CLIP base ----
clip_model, clip_preprocess, clip_tokenizer = load_eva_clip(
    device=device,
    model_name="EVA02-L-14",
    pretrained="merged2b_s4b_b131k",
    to_float32=False,
)

# ---- load your CITYSCAPES finetuned checkpoint ----
ckpt_path = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_gta5_ctxfusion_19cls.pt"
import torch

ckpt = torch.load(ckpt_path, map_location="cpu")

# find state dict
if isinstance(ckpt, dict) and "model" in ckpt:
    sd = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    sd = ckpt["state_dict"]
elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    sd = ckpt["model_state_dict"]
elif isinstance(ckpt, dict) and any(isinstance(v, torch.Tensor) for v in ckpt.values()):
    sd = ckpt  # raw state_dict stored as dict
else:
    raise ValueError(f"Unknown checkpoint format. Keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

missing = clip_model.load_state_dict(sd, strict=False)
print("Loaded. Missing/unexpected:", missing)

# optional: restore logit_scale if saved
if isinstance(ckpt, dict) and "logit_scale_exp" in ckpt:
    with torch.no_grad():
        clip_model.logit_scale.copy_(torch.log(torch.tensor(ckpt["logit_scale_exp"])))
    print("Restored logit_scale from logit_scale_exp.")

print("Loaded finetuned EVA-CLIP checkpoint from:", ckpt_path)

eva = EVACLIPWrapper(
    clip_model,
    clip_preprocess,
    clip_tokenizer,
    device=device,
    combine="add",        # matches your ctx-fusion setup
    embed_dim=1024,
).to(device).eval()
print("EVA wrapper ready.")


Loaded. Missing/unexpected: <All keys matched successfully>
Loaded finetuned EVA-CLIP checkpoint from: /home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_gta5_ctxfusion_19cls.pt
EVA wrapper ready.


In [35]:
from pathlib import Path
import random

CITY = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes")

CS_VAL_IMG_ROOT = CITY / "leftImg8bit" / "val"
CS_VAL_GT_ROOT  = CITY / "gtFine" / "val"   # labelIds live here

city_paths = sorted(CS_VAL_IMG_ROOT.rglob("*_leftImg8bit.png"))
print("Cityscapes val images:", len(city_paths))  # should be 500

# if you want "only 500" explicitly (it already is), keep this:
MAX_EVAL = 500
city_paths = city_paths[:MAX_EVAL]

# optional: shuffle subset reproducibly
# random.seed(42)
# random.shuffle(city_paths)
# city_paths = city_paths[:500]

def gt_labelIds_path_from_img(img_path: Path) -> Path:
    # leftImg8bit/val/<city>/<name>_leftImg8bit.png
    # gtFine/val/<city>/<name>_gtFine_labelIds.png
    city = img_path.parent.name
    return CS_VAL_GT_ROOT / city / img_path.name.replace("_leftImg8bit.png", "_gtFine_labelIds.png")

# quick sanity
print("Example img:", city_paths[0])
print("Example gt :", gt_labelIds_path_from_img(city_paths[0]))
print("GT exists? :", gt_labelIds_path_from_img(city_paths[0]).exists())


Cityscapes val images: 500
Example img: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
Example gt : /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/gtFine/val/frankfurt/frankfurt_000000_000294_gtFine_labelIds.png
GT exists? : True


In [36]:
from pathlib import Path

OUT_ROOT_GT = CITY / "pseudo_cityscapes19_bliptest_GTVAL"
OUT_CLIP_GT = OUT_ROOT_GT / "clip_only_v2"
OUT_CLIPBLIP_GT = OUT_ROOT_GT / "clip_plus_blip_v2"

for d in [OUT_CLIP_GT, OUT_CLIPBLIP_GT]:
    (d / "imgs").mkdir(parents=True, exist_ok=True)
    (d / "lbls").mkdir(parents=True, exist_ok=True)
    (d / "meta").mkdir(parents=True, exist_ok=True)

print("Writing to:", OUT_ROOT_GT)


Writing to: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL


In [35]:
from tqdm import tqdm

ensure_eva_image_on(device)

for img_path in tqdm(city_paths, desc="CLIP-only v2 (Cityscapes val)"):
    out = make_pseudo_clip_only_v2(pil_from_path(img_path), temp_calib=0.07)
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIP_GT, img_path, out["label"], meta)

for img_path in tqdm(city_paths, desc="CLIP+BLIP v2 (Cityscapes val)"):
    out = make_pseudo_clip_plus_blip_v2(pil_from_path(img_path), temp_calib=0.07)
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIPBLIP_GT, img_path, out["label"], meta)

print("Saved to:", OUT_ROOT_GT)


CLIP+BLIP v2 (Cityscapes val): 100%|██████████| 500/500 [1:02:16<00:00,  7.47s/it]

Saved to: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL


In [37]:
import numpy as np
from PIL import Image
from pathlib import Path

IGNORE = 255
NUM_CLASSES = 19

# labelIds -> trainIds LUT
_LUT = np.full(256, IGNORE, dtype=np.uint8)
_map = {7:0,8:1,11:2,12:3,13:4,17:5,19:6,20:7,21:8,22:9,23:10,24:11,25:12,26:13,27:14,28:15,31:16,32:17,33:18}
for k,v in _map.items():
    _LUT[k] = v

def labelIds_to_trainIds(arr_u8: np.ndarray) -> np.ndarray:
    return _LUT[arr_u8]

def _maybe_fix_pred_ids(pr: np.ndarray) -> np.ndarray:
    """
    pr is expected to be trainIds (0..18) or 255.
    If it looks like Cityscapes labelIds (contains keys like 7,8,11,...),
    convert it to trainIds. Otherwise, just clamp invalids to IGNORE.
    """
    pr = pr.astype(np.int64)
    pr_unique = np.unique(pr)
    # Looks like labelIds if it contains typical Cityscapes IDs
    if pr.max() > (NUM_CLASSES - 1) and np.isin(pr_unique, np.array(list(_map.keys()))).any():
        pr = labelIds_to_trainIds(pr.astype(np.uint8)).astype(np.int64)

    # Anything still outside range -> IGNORE
    bad = (pr != IGNORE) & ((pr < 0) | (pr >= NUM_CLASSES))
    if bad.any():
        pr[bad] = IGNORE
    return pr

def confusion_matrix_np(pred: np.ndarray, target: np.ndarray,
                        num_classes=NUM_CLASSES, ignore_index=IGNORE) -> np.ndarray:
    # IMPORTANT: require BOTH gt and pred valid
    k = (
        (target != ignore_index) &
        (pred   != ignore_index) &
        (target >= 0) & (target < num_classes) &
        (pred   >= 0) & (pred   < num_classes)
    )
    if k.sum() == 0:
        return np.zeros((num_classes, num_classes), dtype=np.int64)

    x = target[k].astype(np.int64) * num_classes + pred[k].astype(np.int64)
    binc = np.bincount(x, minlength=num_classes * num_classes)
    return binc[: num_classes * num_classes].reshape(num_classes, num_classes)

def metrics_from_cm_np(cm: np.ndarray):
    eps = 1e-8
    diag = np.diag(cm).astype(np.float64)
    sum_row = cm.sum(axis=1).astype(np.float64)
    sum_col = cm.sum(axis=0).astype(np.float64)
    union = sum_row + sum_col - diag
    iou = (diag + eps) / (union + eps)
    dice = (2*diag + eps) / (sum_row + sum_col + eps)
    miou = float(np.nanmean(iou))
    mdice = float(np.nanmean(dice))
    pixacc = float((diag.sum() + eps) / (cm.sum() + eps))
    return {"miou": miou, "mdice": mdice, "pixel_accuracy": pixacc}

def eval_cityscapes_pseudo(pred_lbl_dir: Path, img_paths):
    pred_lbl_dir = Path(pred_lbl_dir)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    n_used = 0
    n_missing = 0
    overlap_total = 0
    correct_on_overlap = 0
    valid_gt_total = 0
    labeled_on_valid_gt = 0

    for img_path in img_paths:
        pred_path = pred_lbl_dir / img_path.name
        gt_path = gt_labelIds_path_from_img(img_path)  # you already have this

        if not pred_path.exists():
            n_missing += 1
            continue

        pr = np.array(Image.open(pred_path), dtype=np.int64)
        pr = _maybe_fix_pred_ids(pr)

        gt_ids = np.array(Image.open(gt_path), dtype=np.uint8)
        gt = labelIds_to_trainIds(gt_ids).astype(np.int64)

        if pr.shape != gt.shape:
            pr = np.array(
                Image.fromarray(pr.astype(np.uint8)).resize((gt.shape[1], gt.shape[0]), resample=Image.NEAREST),
                dtype=np.int64
            )

        # stats like your earlier function
        valid_gt = (gt != IGNORE)
        labeled  = (pr != IGNORE)
        overlap  = valid_gt & labeled

        valid_gt_total      += int(valid_gt.sum())
        labeled_on_valid_gt += int(overlap.sum())
        if overlap.any():
            correct_on_overlap += int((pr[overlap] == gt[overlap]).sum())
            overlap_total      += int(overlap.sum())

        cm += confusion_matrix_np(pr, gt)
        n_used += 1

    m = metrics_from_cm_np(cm)
    coverage = labeled_on_valid_gt / max(valid_gt_total, 1)
    pixacc_overlap = correct_on_overlap / max(overlap_total, 1)

    print(f"Used {n_used}/{len(img_paths)} preds | missing: {n_missing}")
    print(f"coverage_on_valid_gt: {coverage:.4f} | pixAcc_on_labeled_pixels: {pixacc_overlap:.4f}")
    print(f"mIoU: {m['miou']:.4f} | mDice: {m['mdice']:.4f} | pixAcc(cm): {m['pixel_accuracy']:.4f}")
    m.update({
        "images_used": int(n_used),
        "missing_preds": int(n_missing),
        "coverage_on_valid_gt": float(coverage),
        "pixel_acc_on_labeled_pixels": float(pixacc_overlap),
    })
    return m


In [38]:
m_clip = eval_cityscapes_pseudo(OUT_CLIP_GT / "lbls", city_paths)
m_blip = eval_cityscapes_pseudo(OUT_CLIPBLIP_GT / "lbls", city_paths)

print("\nΔ (BLIP - CLIP):")
print("mIoU:", m_blip["miou"] - m_clip["miou"])
print("mDice:", m_blip["mdice"] - m_clip["mdice"])
print("pixAcc(cm):", m_blip["pixel_accuracy"] - m_clip["pixel_accuracy"])
print("coverage:", m_blip["coverage_on_valid_gt"] - m_clip["coverage_on_valid_gt"])


Used 500/500 preds | missing: 0
coverage_on_valid_gt: 0.0540 | pixAcc_on_labeled_pixels: 0.4578
mIoU: 0.1554 | mDice: 0.2116 | pixAcc(cm): 0.4578
Used 500/500 preds | missing: 0
coverage_on_valid_gt: 0.0484 | pixAcc_on_labeled_pixels: 0.1348
mIoU: 0.1239 | mDice: 0.1802 | pixAcc(cm): 0.1348

Δ (BLIP - CLIP):
mIoU: -0.031517133860957294
mDice: -0.031431698111114986
pixAcc(cm): -0.32303779237660624
coverage: -0.005630593125369364


In [40]:
CITY = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes")

GT_LBLS = str(CITY / "seg_data_trainId" / "val_lbls")   # <-- Cityscapes GT in trainId space (flat)
CLIP_LBLS = str(OUT_CLIP_GT / "lbls")
CLIP_BLIP_LBLS = str(OUT_CLIPBLIP_GT / "lbls")

print("GT_LBLS:", GT_LBLS)
print("CLIP_LBLS:", CLIP_LBLS)
print("CLIP_BLIP_LBLS:", CLIP_BLIP_LBLS)


GT_LBLS: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/seg_data_trainId/val_lbls
CLIP_LBLS: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL/clip_only_v2/lbls
CLIP_BLIP_LBLS: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL/clip_plus_blip_v2/lbls


In [41]:
gt_names = {p.name for p in Path(GT_LBLS).glob("*.png")}
pr_names = {p.name for p in Path(CLIP_LBLS).glob("*.png")}

common = sorted(gt_names & pr_names)
print("GT:", len(gt_names), "PR:", len(pr_names), "COMMON:", len(common))
print("example common:", common[:5])
print("example GT-only:", sorted(gt_names - pr_names)[:5])
print("example PR-only:", sorted(pr_names - gt_names)[:5])


GT: 500 PR: 500 COMMON: 500
example common: ['frankfurt_000000_000294_leftImg8bit.png', 'frankfurt_000000_000576_leftImg8bit.png', 'frankfurt_000000_001016_leftImg8bit.png', 'frankfurt_000000_001236_leftImg8bit.png', 'frankfurt_000000_001751_leftImg8bit.png']
example GT-only: []
example PR-only: []


In [42]:
print("[PSEUDO vs GT] CLIP-only:", eval_pseudo_vs_gt_common(CLIP_LBLS, GT_LBLS))
print("[PSEUDO vs GT] CLIP+BLIP:", eval_pseudo_vs_gt_common(CLIP_BLIP_LBLS, GT_LBLS))


common files: 500
[PSEUDO vs GT] CLIP-only: {'images_used': 500, 'coverage_on_valid_gt': 0.06291551772627346, 'pixel_acc_on_labeled_pixels': 0.4631011209297881, 'mIoU_on_labeled_pixels_present_classes': 0.18450795891221028}
common files: 500
[PSEUDO vs GT] CLIP+BLIP: {'images_used': 500, 'coverage_on_valid_gt': 0.10234943365465775, 'pixel_acc_on_labeled_pixels': 0.6162995859106903, 'mIoU_on_labeled_pixels_present_classes': 0.294966490359123}


In [43]:
margins = []
pcals = []
logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

# use Cityscapes images now
for img_path in city_paths[:5]:
    pil_img = pil_from_path(img_path)
    masks = select_top_sam_masks(sam_on_pil(pil_img), top_k=25, min_area=800)
    clip_results = classify_masks_with_clip(pil_img, masks)

    for r in clip_results:
        logits = r["logits"]
        if torch.is_tensor(logits):
            logits = logits.detach().cpu().numpy()

        top1, top2, margin_cos, pcal_top1, ent = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=0.07
        )
        margins.append(margin_cos)
        pcals.append(pcal_top1)

print("margin_cos: min/med/max", np.min(margins), np.median(margins), np.max(margins))
print("pcal_top1:  min/med/max", np.min(pcals), np.median(pcals), np.max(pcals))


margin_cos: min/med/max 0.007962101139128208 0.11610172688961029 0.33583885431289673
pcal_top1:  min/med/max 0.19078589975833893 0.6015599370002747 0.9718407988548279


## lora cityscapes

In [33]:
# ---- EVA-CLIP base ----
clip_model, clip_preprocess, clip_tokenizer = load_eva_clip(
    device=device,
    model_name="EVA02-L-14",
    pretrained="merged2b_s4b_b131k",
    to_float32=False,
)

# ---- load your CITYSCAPES finetuned checkpoint ----
ckpt_path = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_gta5_ctxfusion_19cls_MERGED.pt"
import torch

ckpt = torch.load(ckpt_path, map_location="cpu")

# find state dict
if isinstance(ckpt, dict) and "model" in ckpt:
    sd = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    sd = ckpt["state_dict"]
elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    sd = ckpt["model_state_dict"]
elif isinstance(ckpt, dict) and any(isinstance(v, torch.Tensor) for v in ckpt.values()):
    sd = ckpt  # raw state_dict stored as dict
else:
    raise ValueError(f"Unknown checkpoint format. Keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

missing = clip_model.load_state_dict(sd, strict=False)
print("Loaded. Missing/unexpected:", missing)

# optional: restore logit_scale if saved
if isinstance(ckpt, dict) and "logit_scale_exp" in ckpt:
    with torch.no_grad():
        clip_model.logit_scale.copy_(torch.log(torch.tensor(ckpt["logit_scale_exp"])))
    print("Restored logit_scale from logit_scale_exp.")

print("Loaded finetuned EVA-CLIP checkpoint from:", ckpt_path)

eva = EVACLIPWrapper(
    clip_model,
    clip_preprocess,
    clip_tokenizer,
    device=device,
    combine="add",        # matches your ctx-fusion setup
    embed_dim=1024,
).to(device).eval()
print("EVA wrapper ready.")

from pathlib import Path
import random

CITY = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes")

CS_VAL_IMG_ROOT = CITY / "leftImg8bit" / "val"
CS_VAL_GT_ROOT  = CITY / "gtFine" / "val"   # labelIds live here

city_paths = sorted(CS_VAL_IMG_ROOT.rglob("*_leftImg8bit.png"))
print("Cityscapes val images:", len(city_paths))  # should be 500

# if you want "only 500" explicitly (it already is), keep this:
MAX_EVAL = 500
city_paths = city_paths[:MAX_EVAL]

# optional: shuffle subset reproducibly
# random.seed(42)
# random.shuffle(city_paths)
# city_paths = city_paths[:500]

def gt_labelIds_path_from_img(img_path: Path) -> Path:
    # leftImg8bit/val/<city>/<name>_leftImg8bit.png
    # gtFine/val/<city>/<name>_gtFine_labelIds.png
    city = img_path.parent.name
    return CS_VAL_GT_ROOT / city / img_path.name.replace("_leftImg8bit.png", "_gtFine_labelIds.png")

# quick sanity
print("Example img:", city_paths[0])
print("Example gt :", gt_labelIds_path_from_img(city_paths[0]))
print("GT exists? :", gt_labelIds_path_from_img(city_paths[0]).exists())

from pathlib import Path

OUT_ROOT_GT = CITY / "pseudo_cityscapes19_bliptest_GTVAL"
OUT_CLIP_GT = OUT_ROOT_GT / "clip_only_v2"
OUT_CLIPBLIP_GT = OUT_ROOT_GT / "clip_plus_blip_v2"

for d in [OUT_CLIP_GT, OUT_CLIPBLIP_GT]:
    (d / "imgs").mkdir(parents=True, exist_ok=True)
    (d / "lbls").mkdir(parents=True, exist_ok=True)
    (d / "meta").mkdir(parents=True, exist_ok=True)

print("Writing to:", OUT_ROOT_GT)

from tqdm import tqdm

ensure_eva_image_on(device)

for img_path in tqdm(city_paths, desc="CLIP-only v2 (Cityscapes val)"):
    out = make_pseudo_clip_only_v2(pil_from_path(img_path), temp_calib=0.07)
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIP_GT, img_path, out["label"], meta)

for img_path in tqdm(city_paths, desc="CLIP+BLIP v2 (Cityscapes val)"):
    out = make_pseudo_clip_plus_blip_v2(pil_from_path(img_path), temp_calib=0.07)
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIPBLIP_GT, img_path, out["label"], meta)

print("Saved to:", OUT_ROOT_GT)

import numpy as np
from PIL import Image
from pathlib import Path

IGNORE = 255
NUM_CLASSES = 19

# labelIds -> trainIds LUT
_LUT = np.full(256, IGNORE, dtype=np.uint8)
_map = {7:0,8:1,11:2,12:3,13:4,17:5,19:6,20:7,21:8,22:9,23:10,24:11,25:12,26:13,27:14,28:15,31:16,32:17,33:18}
for k,v in _map.items():
    _LUT[k] = v

def labelIds_to_trainIds(arr_u8: np.ndarray) -> np.ndarray:
    return _LUT[arr_u8]

def _maybe_fix_pred_ids(pr: np.ndarray) -> np.ndarray:
    """
    pr is expected to be trainIds (0..18) or 255.
    If it looks like Cityscapes labelIds (contains keys like 7,8,11,...),
    convert it to trainIds. Otherwise, just clamp invalids to IGNORE.
    """
    pr = pr.astype(np.int64)
    pr_unique = np.unique(pr)
    # Looks like labelIds if it contains typical Cityscapes IDs
    if pr.max() > (NUM_CLASSES - 1) and np.isin(pr_unique, np.array(list(_map.keys()))).any():
        pr = labelIds_to_trainIds(pr.astype(np.uint8)).astype(np.int64)

    # Anything still outside range -> IGNORE
    bad = (pr != IGNORE) & ((pr < 0) | (pr >= NUM_CLASSES))
    if bad.any():
        pr[bad] = IGNORE
    return pr

def confusion_matrix_np(pred: np.ndarray, target: np.ndarray,
                        num_classes=NUM_CLASSES, ignore_index=IGNORE) -> np.ndarray:
    # IMPORTANT: require BOTH gt and pred valid
    k = (
        (target != ignore_index) &
        (pred   != ignore_index) &
        (target >= 0) & (target < num_classes) &
        (pred   >= 0) & (pred   < num_classes)
    )
    if k.sum() == 0:
        return np.zeros((num_classes, num_classes), dtype=np.int64)

    x = target[k].astype(np.int64) * num_classes + pred[k].astype(np.int64)
    binc = np.bincount(x, minlength=num_classes * num_classes)
    return binc[: num_classes * num_classes].reshape(num_classes, num_classes)

def metrics_from_cm_np(cm: np.ndarray):
    eps = 1e-8
    diag = np.diag(cm).astype(np.float64)
    sum_row = cm.sum(axis=1).astype(np.float64)
    sum_col = cm.sum(axis=0).astype(np.float64)
    union = sum_row + sum_col - diag
    iou = (diag + eps) / (union + eps)
    dice = (2*diag + eps) / (sum_row + sum_col + eps)
    miou = float(np.nanmean(iou))
    mdice = float(np.nanmean(dice))
    pixacc = float((diag.sum() + eps) / (cm.sum() + eps))
    return {"miou": miou, "mdice": mdice, "pixel_accuracy": pixacc}

def eval_cityscapes_pseudo(pred_lbl_dir: Path, img_paths):
    pred_lbl_dir = Path(pred_lbl_dir)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    n_used = 0
    n_missing = 0
    overlap_total = 0
    correct_on_overlap = 0
    valid_gt_total = 0
    labeled_on_valid_gt = 0

    for img_path in img_paths:
        pred_path = pred_lbl_dir / img_path.name
        gt_path = gt_labelIds_path_from_img(img_path)  # you already have this

        if not pred_path.exists():
            n_missing += 1
            continue

        pr = np.array(Image.open(pred_path), dtype=np.int64)
        pr = _maybe_fix_pred_ids(pr)

        gt_ids = np.array(Image.open(gt_path), dtype=np.uint8)
        gt = labelIds_to_trainIds(gt_ids).astype(np.int64)

        if pr.shape != gt.shape:
            pr = np.array(
                Image.fromarray(pr.astype(np.uint8)).resize((gt.shape[1], gt.shape[0]), resample=Image.NEAREST),
                dtype=np.int64
            )

        # stats like your earlier function
        valid_gt = (gt != IGNORE)
        labeled  = (pr != IGNORE)
        overlap  = valid_gt & labeled

        valid_gt_total      += int(valid_gt.sum())
        labeled_on_valid_gt += int(overlap.sum())
        if overlap.any():
            correct_on_overlap += int((pr[overlap] == gt[overlap]).sum())
            overlap_total      += int(overlap.sum())

        cm += confusion_matrix_np(pr, gt)
        n_used += 1

    m = metrics_from_cm_np(cm)
    coverage = labeled_on_valid_gt / max(valid_gt_total, 1)
    pixacc_overlap = correct_on_overlap / max(overlap_total, 1)

    print(f"Used {n_used}/{len(img_paths)} preds | missing: {n_missing}")
    print(f"coverage_on_valid_gt: {coverage:.4f} | pixAcc_on_labeled_pixels: {pixacc_overlap:.4f}")
    print(f"mIoU: {m['miou']:.4f} | mDice: {m['mdice']:.4f} | pixAcc(cm): {m['pixel_accuracy']:.4f}")
    m.update({
        "images_used": int(n_used),
        "missing_preds": int(n_missing),
        "coverage_on_valid_gt": float(coverage),
        "pixel_acc_on_labeled_pixels": float(pixacc_overlap),
    })
    return m

m_clip = eval_cityscapes_pseudo(OUT_CLIP_GT / "lbls", city_paths)
m_blip = eval_cityscapes_pseudo(OUT_CLIPBLIP_GT / "lbls", city_paths)

print("\nΔ (BLIP - CLIP):")
print("mIoU:", m_blip["miou"] - m_clip["miou"])
print("mDice:", m_blip["mdice"] - m_clip["mdice"])
print("pixAcc(cm):", m_blip["pixel_accuracy"] - m_clip["pixel_accuracy"])
print("coverage:", m_blip["coverage_on_valid_gt"] - m_clip["coverage_on_valid_gt"])

CITY = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes")

GT_LBLS = str(CITY / "seg_data_trainId" / "val_lbls")   # <-- Cityscapes GT in trainId space (flat)
CLIP_LBLS = str(OUT_CLIP_GT / "lbls")
CLIP_BLIP_LBLS = str(OUT_CLIPBLIP_GT / "lbls")

print("GT_LBLS:", GT_LBLS)
print("CLIP_LBLS:", CLIP_LBLS)
print("CLIP_BLIP_LBLS:", CLIP_BLIP_LBLS)

gt_names = {p.name for p in Path(GT_LBLS).glob("*.png")}
pr_names = {p.name for p in Path(CLIP_LBLS).glob("*.png")}

common = sorted(gt_names & pr_names)
print("GT:", len(gt_names), "PR:", len(pr_names), "COMMON:", len(common))
print("example common:", common[:5])
print("example GT-only:", sorted(gt_names - pr_names)[:5])
print("example PR-only:", sorted(pr_names - gt_names)[:5])

print("[PSEUDO vs GT] CLIP-only:", eval_pseudo_vs_gt_common(CLIP_LBLS, GT_LBLS))
print("[PSEUDO vs GT] CLIP+BLIP:", eval_pseudo_vs_gt_common(CLIP_BLIP_LBLS, GT_LBLS))

margins = []
pcals = []
logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

# use Cityscapes images now
for img_path in city_paths[:5]:
    pil_img = pil_from_path(img_path)
    masks = select_top_sam_masks(sam_on_pil(pil_img), top_k=25, min_area=800)
    clip_results = classify_masks_with_clip(pil_img, masks)

    for r in clip_results:
        logits = r["logits"]
        if torch.is_tensor(logits):
            logits = logits.detach().cpu().numpy()

        top1, top2, margin_cos, pcal_top1, ent = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=0.07
        )
        margins.append(margin_cos)
        pcals.append(pcal_top1)

print("margin_cos: min/med/max", np.min(margins), np.median(margins), np.max(margins))
print("pcal_top1:  min/med/max", np.min(pcals), np.median(pcals), np.max(pcals))


Loaded. Missing/unexpected: <All keys matched successfully>
Loaded finetuned EVA-CLIP checkpoint from: /home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_gta5_ctxfusion_19cls_MERGED.pt
EVA wrapper ready.
Cityscapes val images: 500
Example img: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
Example gt : /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/gtFine/val/frankfurt/frankfurt_000000_000294_gtFine_labelIds.png
GT exists? : True
Writing to: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL


CLIP+BLIP v2 (Cityscapes val): 100%|██████████| 500/500 [1:00:54<00:00,  7.31s/it]


Saved to: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL
Used 500/500 preds | missing: 0
coverage_on_valid_gt: 0.0370 | pixAcc_on_labeled_pixels: 0.8196
mIoU: 0.2287 | mDice: 0.2607 | pixAcc(cm): 0.8196
Used 500/500 preds | missing: 0
coverage_on_valid_gt: 0.0400 | pixAcc_on_labeled_pixels: 0.1624
mIoU: 0.1447 | mDice: 0.1981 | pixAcc(cm): 0.1624

Δ (BLIP - CLIP):
mIoU: -0.08401475345573403
mDice: -0.0626463165700174
pixAcc(cm): -0.6572349208627339
coverage: 0.0030033047676097657
GT_LBLS: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/seg_data_trainId/val_lbls
CLIP_LBLS: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL/clip_only_v2/lbls
CLIP_BLIP_LBLS: /home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes/pseudo_cityscapes19_bliptest_GTVAL/clip_plus_blip_v2/lbls
GT: 500 PR: 500 COMMON: 500
example common: ['frankfurt_000000_000294_leftImg8bit.png', 'frankfurt_000

NameError: name 'eval_pseudo_vs_gt_common' is not defined

## Waste data

In [30]:
# ---------------------------
# Knobs (keep these GLOBAL so both clip_only_v2 and clip_plus_blip_v2 share them)
# ---------------------------

# shared (must match clip_only_v2)
MAX_MASKS_PER_IMAGE  = 15
CLIP_KEEP_PCAL       = 0.85
CLIP_KEEP_MARGIN     = 0.18
IGNORE_LABEL         = 255

# BLIP routing (how much BLIP gets to try)
MAX_BLIP_PER_IMAGE          = 6     # rescue (from non-confident CLIP)
BLIP_OVERRIDE_MAX_PER_IMAGE = 3     # override (from CLIP-kept but ambiguous)

# BLIP accept / scoring
BLIP_MIN_SIM_ACCEPT   = 0.62        # rescue accept threshold
BLIP_SCORE_BOOST      = 0.10        # small boost so BLIP can win overlaps sometimes

# BLIP override behavior
BLIP_OVERRIDE_SIM_MIN = 0.66        # require stronger BLIP to override
BLIP_OVERRIDE_BONUS   = 0.20        # extra bonus only when overriding CLIP
BLIP_OVERRIDE_REQUIRE_TOP2 = False  # if True: only override if BLIP == CLIP top2[1]

# "ambiguous CLIP-kept" definition (controls how often override is attempted)
CLIP_OVERRIDE_MARGIN_SLACK = 0.06   # route to override if margin < (KEEP_MARGIN + slack)
CLIP_OVERRIDE_ENTROPY_MIN  = 1.60   # or if entropy is high


def _bbox_area_frac(bbox, H, W):
    x0,y0,x1,y1 = bbox
    return float(max(0, x1-x0) * max(0, y1-y0) / max(1, H*W))


@torch.no_grad()
def make_pseudo_clip_plus_blip_v2(pil_img: Image.Image, temp_calib=0.07):
    """
    v2: matches your clip_only_v2 pipeline:
      - SAM -> prune_sam_masks_with_bg -> masks_keep + hard_ignore + info
      - CLIP strict keep using (CLIP_KEEP_PCAL, CLIP_KEEP_MARGIN)
      - BLIP rescues non-confident masks (MAX_BLIP_PER_IMAGE)
      - BLIP can override some CLIP-kept-but-ambiguous masks (BLIP_OVERRIDE_MAX_PER_IMAGE)
      - build_semantic_label_map(..., hard_ignore_masks=hard_ignore)
    """
    ensure_eva_image_on(device)

    W, H = pil_img.size

    # 1) SAM + your background/fog pruning
    masks_bool = sam_on_pil(pil_img)
    masks_keep, hard_ignore, info = prune_sam_masks_with_bg(pil_img, masks_bool)

    # 2) CLIP logits for kept masks
    clip_results = classify_masks_with_clip(pil_img, masks_keep)
    logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

    clip_instances_by_mask = {}      # mask_index -> instance (kept by CLIP)
    blip_rescue_candidates = []      # from non-confident CLIP
    blip_override_candidates = []    # from CLIP-kept but ambiguous

    # helper: build a BLIP crop robustly
    def _make_blip_crop(bbox, m_bool):
        x0,y0,x1,y1 = bbox
        mask_crop_bool = m_bool[y0:y1, x0:x1].astype(bool)

        # preferred: your focused crop if it exists
        if "make_blip_focus_crop" in globals():
            try:
                return make_blip_focus_crop(pil_img, bbox, m_bool, bg_mode="gray", blur_bg=True)
            except Exception:
                pass

        # fallback: soft-mask crop
        crop = crop_image(pil_img, bbox)
        return apply_soft_mask_weighting(crop, mask_crop_bool)

    # 3) Decide CLIP-kept vs BLIP routing
    for r in clip_results:
        midx = int(r.get("mask_index", -1))
        bbox = r.get("bbox", None)
        if bbox is None:
            continue

        x0,y0,x1,y1 = bbox
        m = r["mask"].astype(bool)
        logits = r["logits"]  # IMPORTANT: classify_masks_with_clip must include this

        top1, top2, margin_cos, pcal_top1, entropy = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=temp_calib
        )

        label1 = CAT_NAMES[int(top1)]
        label2 = CAT_NAMES[int(top2)]
        frac = _bbox_area_frac(bbox, H, W)

        confident = (pcal_top1 >= CLIP_KEEP_PCAL) and (margin_cos >= CLIP_KEEP_MARGIN)

        if confident:
            # keep CLIP
            clip_instances_by_mask[midx] = {
                "mask_index": midx,
                "mask": m,
                "class_id": int(top1),
                "score": float(pcal_top1),
                "bbox": bbox,
                "src": "clip",
                "pcal": float(pcal_top1),
                "margin": float(margin_cos),
                "entropy": float(entropy),
                "top2": (label1, label2),
            }

            # route some "kept but ambiguous" to BLIP for override attempt
            ambiguous_kept = (margin_cos < (CLIP_KEEP_MARGIN + CLIP_OVERRIDE_MARGIN_SLACK)) or (entropy > CLIP_OVERRIDE_ENTROPY_MIN)
            if ambiguous_kept and (len(blip_override_candidates) < BLIP_OVERRIDE_MAX_PER_IMAGE * 6):
                blip_override_candidates.append({
                    "mask_index": midx,
                    "mask": m,
                    "bbox": bbox,
                    "crop_blip": _make_blip_crop(bbox, m),
                    "clip_label": label1,
                    "clip_top2": (label1, label2),
                    "margin_cos": float(margin_cos),
                    "pcal_top1": float(pcal_top1),
                    "entropy": float(entropy),
                    "bbox_frac": float(frac),
                    "mode": "override",
                })

        else:
            # route to BLIP rescue if your gating says yes
            if should_trigger_blip(label1, label2, margin_cos, pcal_top1, entropy, frac):
                blip_rescue_candidates.append({
                    "mask_index": midx,
                    "mask": m,
                    "bbox": bbox,
                    "crop_blip": _make_blip_crop(bbox, m),
                    "clip_top2": (label1, label2),
                    "margin_cos": float(margin_cos),
                    "pcal_top1": float(pcal_top1),
                    "entropy": float(entropy),
                    "bbox_frac": float(frac),
                    "mode": "rescue",
                })

    # 4) Limit BLIP work (pick most uncertain first)
    blip_rescue_candidates = sorted(
        blip_rescue_candidates, key=lambda e: (e["margin_cos"], -e["entropy"])
    )[:MAX_BLIP_PER_IMAGE]

    blip_override_candidates = sorted(
        blip_override_candidates, key=lambda e: (e["margin_cos"], -e["entropy"])
    )[:BLIP_OVERRIDE_MAX_PER_IMAGE]

    blip_candidates = blip_rescue_candidates + blip_override_candidates

    # 5) Run BLIP captions
    crops = [e["crop_blip"] for e in blip_candidates]
    captions = blip_caption(crops, batch_size=4, max_new_tokens=15) if crops else []

    blip_instances = []
    blip_rejected = []
    overrides_done = 0
    rescue_done = 0

    for e, cap in zip(blip_candidates, captions):
        mapping = map_caption_to_class_gpu(cap)
        mapped_idx   = int(mapping["best_idx"])
        mapped_label = mapping["best_label"]
        mapped_sim   = float(mapping["best_sim"])

        if e["mode"] == "rescue":
            ok = (
                (mapped_sim >= BLIP_MIN_SIM_ACCEPT) and
                should_accept_blip(mapped_label, mapped_sim, e["clip_top2"][0], e["clip_top2"][1])
            )

            if ok:
                score = min(1.0, mapped_sim + BLIP_SCORE_BOOST)
                blip_instances.append({
                    "mask_index": e["mask_index"],
                    "mask": e["mask"],
                    "class_id": mapped_idx,
                    "score": score,
                    "bbox": e["bbox"],
                    "src": "blip_rescue",
                    "caption": cap,
                    "mapped_label": mapped_label,
                    "mapped_sim": mapped_sim,
                    "clip_top2": e["clip_top2"],
                })
                rescue_done += 1
            else:
                blip_rejected.append({
                    "mode": "rescue",
                    "caption": cap,
                    "mapped": mapped_label,
                    "sim": mapped_sim,
                    "clip_top2": e["clip_top2"],
                })

        else:
            # override mode: BLIP can replace CLIP for SAME mask_index
            clip_label = e["clip_label"]
            clip_top2  = e["clip_top2"]

            override_ok = (mapped_sim >= BLIP_OVERRIDE_SIM_MIN) and (mapped_label != clip_label)

            if BLIP_OVERRIDE_REQUIRE_TOP2:
                override_ok = override_ok and (mapped_label == clip_top2[1])

            if override_ok:
                # remove CLIP instance for this mask_index
                clip_instances_by_mask.pop(e["mask_index"], None)

                score = min(1.0, mapped_sim + BLIP_SCORE_BOOST + BLIP_OVERRIDE_BONUS)
                blip_instances.append({
                    "mask_index": e["mask_index"],
                    "mask": e["mask"],
                    "class_id": mapped_idx,
                    "score": score,
                    "bbox": e["bbox"],
                    "src": "blip_override",
                    "caption": cap,
                    "mapped_label": mapped_label,
                    "mapped_sim": mapped_sim,
                    "clip_prev": clip_label,
                    "clip_top2": clip_top2,
                })
                overrides_done += 1
            else:
                blip_rejected.append({
                    "mode": "override",
                    "caption": cap,
                    "mapped": mapped_label,
                    "sim": mapped_sim,
                    "clip_label": clip_label,
                    "clip_top2": clip_top2,
                })

    # 6) Merge + cap final masks
    clip_instances = list(clip_instances_by_mask.values())
    merged = clip_instances + blip_instances
    merged = sorted(merged, key=lambda x: x["score"], reverse=True)[:MAX_MASKS_PER_IMAGE]

    # 7) Build label map (IMPORTANT: include hard_ignore masks)
    lbl = build_semantic_label_map(
        H, W, merged,
        ignore_label=IGNORE_LABEL,
        hard_ignore_masks=hard_ignore
    )

    meta = {
        "mode": "clip_plus_blip_v2",
        "n_sam_raw": len(masks_bool),
        "n_sam_keep": len(masks_keep),
        "n_hard_ignore": len(hard_ignore),
        **info,

        "n_clip_results": len(clip_results),
        "n_clip_kept": len(clip_instances),

        "n_blip_rescue_candidates": len(blip_rescue_candidates),
        "n_blip_override_candidates": len(blip_override_candidates),
        "n_blip_candidates": len(blip_candidates),

        "n_blip_accepted_total": len(blip_instances),
        "n_blip_rescues_done": rescue_done,
        "n_overrides_done": overrides_done,

        "n_final_merged": len(merged),

        "temp_calib": temp_calib,

        # snapshot knobs for reproducibility
        "MAX_MASKS_PER_IMAGE": MAX_MASKS_PER_IMAGE,
        "CLIP_KEEP_PCAL": CLIP_KEEP_PCAL,
        "CLIP_KEEP_MARGIN": CLIP_KEEP_MARGIN,
        "MAX_BLIP_PER_IMAGE": MAX_BLIP_PER_IMAGE,
        "BLIP_OVERRIDE_MAX_PER_IMAGE": BLIP_OVERRIDE_MAX_PER_IMAGE,
        "BLIP_MIN_SIM_ACCEPT": BLIP_MIN_SIM_ACCEPT,
        "BLIP_SCORE_BOOST": BLIP_SCORE_BOOST,
        "BLIP_OVERRIDE_SIM_MIN": BLIP_OVERRIDE_SIM_MIN,
        "BLIP_OVERRIDE_BONUS": BLIP_OVERRIDE_BONUS,
        "BLIP_OVERRIDE_REQUIRE_TOP2": BLIP_OVERRIDE_REQUIRE_TOP2,
        "CLIP_OVERRIDE_MARGIN_SLACK": CLIP_OVERRIDE_MARGIN_SLACK,
        "CLIP_OVERRIDE_ENTROPY_MIN": CLIP_OVERRIDE_ENTROPY_MIN,

        # small preview for debugging (keep tiny!)
        "blip_rejected_preview": blip_rejected[:5],
    }

    return {"label": lbl, "meta": meta}


In [31]:
import re
import numpy as np
import torch

# =========================
# Waste ontology (9 classes, with background)
# =========================
IOSB_CLASSES_9 = [
    "background",
    "bottle",
    "bag_film",
    "cup_tray",
    "lid_cap",
    "carton",
    "can",
    "foam",
    "other_packaging",
]
TRAIN_CLASSES_8 = IOSB_CLASSES_9[1:]  # 8 classes (train ids 0..7 after cid-1)

CAT_NAMES = IOSB_CLASSES_9[:]         # <-- use this everywhere (CLIP, BLIP, eval)
NUM_CLASSES = len(CAT_NAMES)
IGNORE = 255

print("NUM_CLASSES:", NUM_CLASSES)
print("CAT_NAMES:", CAT_NAMES)

# =========================
# CLIP prompt engineering
# =========================
IOSB_SYNONYMS = {
  "bottle": [
    "a plastic bottle", "a PET bottle", "a water bottle", "a soda bottle"
  ],
  "bag_film": [
    "a plastic bag", "plastic film", "shrink wrap", "wrapping film", "plastic wrapper"
  ],
  "cup_tray": [
    "a plastic cup", "a food tray", "a plastic tray", "a takeaway container",
    "a food container", "a clamshell container"
  ],
  "lid_cap": [
    "a bottle cap", "a plastic cap", "a lid", "a container lid"
  ],
  "carton": [
    "a beverage carton", "a milk carton", "a juice carton", "a liquid carton (Tetra Pak)"
  ],
  "can": [
    "a metal can", "an aluminum can", "a soda can", "a tin can"
  ],
  "foam": [
    "foam packaging", "styrofoam", "expanded polystyrene foam", "foam tray"
  ],
  "other_packaging": [
    "other packaging", "miscellaneous packaging", "unknown packaging item"
  ],
}

IOSB_TEMPLATES = [
  "a waste sorting image containing {}",
  "an industrial waste stream showing {}",
  "a conveyor-belt scene with {}",
  "a waste item: {}",
  "an image of {} in a waste sorting setting",
  "a piece of {} packaging on a conveyor belt",
]

def build_clip_text_prompts(cat_names=CAT_NAMES,
                            synonyms=IOSB_SYNONYMS,
                            templates=IOSB_TEMPLATES):
    """
    Returns:
      prompts: list[str] of length sum_k (#templates * #synonyms_per_class)
      prompt_to_class: list[int] same length (class id for each prompt)
    """
    prompts = []
    prompt_to_class = []

    # background prompts (keep few, simple)
    bg_phrases = [
        "background", "conveyor belt", "empty belt", "belt surface", "no object"
    ]
    for ph in bg_phrases:
        for t in templates[:3]:
            prompts.append(t.format(ph))
            prompt_to_class.append(0)

    for cid, cname in enumerate(cat_names):
        if cname == "background":
            continue
        syns = synonyms.get(cname, [cname.replace("_", " ")])
        for s in syns:
            for t in templates:
                prompts.append(t.format(s))
                prompt_to_class.append(cid)

    return prompts, np.array(prompt_to_class, dtype=np.int64)

CLIP_PROMPTS, CLIP_PROMPT2CID = build_clip_text_prompts()
print("Total CLIP prompts:", len(CLIP_PROMPTS))
print("Prompts per class (approx):", {c: int((CLIP_PROMPT2CID==i).sum()) for i,c in enumerate(CAT_NAMES)})

# =========================
# BLIP -> class mapping
# =========================

# 1) Keyword rules (fast, deterministic)
BLIP_KEYWORDS = {
    "bottle": [
        r"\bbottle\b", r"\bpet\b", r"\bwater bottle\b", r"\bsoda bottle\b"
    ],
    "bag_film": [
        r"\bbag\b", r"\bplastic bag\b", r"\bfilm\b", r"\bwrap\b", r"\bwrapper\b", r"\bshrink wrap\b"
    ],
    "cup_tray": [
        r"\bcup\b", r"\btray\b", r"\bcontainer\b", r"\bclamshell\b", r"\btakeaway\b", r"\bfood tray\b"
    ],
    "lid_cap": [
        r"\bcap\b", r"\blid\b", r"\bbottle cap\b", r"\bcontainer lid\b"
    ],
    "carton": [
        r"\bcarton\b", r"\btetra\b", r"\btetra pak\b", r"\bmilk carton\b", r"\bjuice carton\b"
    ],
    "can": [
        r"\bcan\b", r"\baluminum can\b", r"\btin can\b", r"\bmetal can\b"
    ],
    "foam": [
        r"\bfoam\b", r"\bstyrofoam\b", r"\bpolystyrene\b", r"\beps\b"
    ],
    "other_packaging": [
        r"\bpackaging\b", r"\bplastic\b", r"\bwrapper\b", r"\bcontainer\b"
    ],
}

# background triggers (if BLIP says belt / nothing / empty)
BLIP_BG = [
    r"\bconveyor\b", r"\bbelt\b", r"\bbackground\b", r"\bempty\b", r"\bno object\b"
]

def blip_caption_to_class_rule(caption: str):
    """
    Returns cid if confident by rules, else None.
    """
    if caption is None:
        return None
    s = caption.lower()

    # background check
    if any(re.search(p, s) for p in BLIP_BG):
        return 0

    hits = []
    for cname, pats in BLIP_KEYWORDS.items():
        if any(re.search(p, s) for p in pats):
            hits.append(cname)

    if len(hits) == 1:
        return CAT_NAMES.index(hits[0])
    if len(hits) == 0:
        return None
    # multiple hits -> ambiguous
    return None

# 2) Fallback: map BLIP caption to class via EVA-CLIP text similarity
#    (this uses your fine-tuned EVA-CLIP checkpoint)
@torch.no_grad()
def blip_caption_to_class_clip(caption: str, eva, cat_names=CAT_NAMES, temp_calib=0.07):
    """
    Uses EVA-CLIP text embedding similarity between the caption and class prompts.
    Returns (cid, confidence_like_score).
    """
    if caption is None or len(caption.strip()) == 0:
        return 0, 0.0

    # Encode caption as text
    # NOTE: depends on your EVACLIPWrapper API. Commonly: eva.encode_text(list_of_strings)
    # If your wrapper differs, adjust here.
    txt_emb = eva.encode_text([caption])  # shape (1, D)
    if torch.is_tensor(txt_emb):
        txt_emb = txt_emb / (txt_emb.norm(dim=-1, keepdim=True) + 1e-8)

    # Encode class prompts once and cache for speed
    # We'll create a cache on the eva object
    if not hasattr(eva, "_iosb_prompt_cache"):
        embs = eva.encode_text(CLIP_PROMPTS)  # (P, D)
        if torch.is_tensor(embs):
            embs = embs / (embs.norm(dim=-1, keepdim=True) + 1e-8)
        eva._iosb_prompt_cache = embs
        eva._iosb_prompt2cid = torch.from_numpy(CLIP_PROMPT2CID).to(embs.device)

    embs = eva._iosb_prompt_cache
    p2c  = eva._iosb_prompt2cid

    # cosine sim
    sims = (txt_emb @ embs.T).squeeze(0)  # (P,)
    # aggregate per class by max sim over prompts
    cids = torch.unique(p2c)
    best_sim = []
    for cid in range(NUM_CLASSES):
        mask = (p2c == cid)
        best_sim.append(sims[mask].max())
    best_sim = torch.stack(best_sim)  # (C,)

    cid = int(best_sim.argmax().item())
    # a simple confidence proxy: softmax over classes
    probs = torch.softmax(best_sim / temp_calib, dim=0)
    conf = float(probs[cid].item())
    return cid, conf

def blip_caption_to_class(caption: str, eva, rule_first=True):
    """
    Unified BLIP->class mapping:
      - Try deterministic keyword rule
      - Else fallback to EVA-CLIP text similarity
    Returns: (cid, source, conf)
    """
    if rule_first:
        cid = blip_caption_to_class_rule(caption)
        if cid is not None:
            return cid, "rule", 1.0

    cid, conf = blip_caption_to_class_clip(caption, eva)
    return cid, "clip_text", conf


NUM_CLASSES: 9
CAT_NAMES: ['background', 'bottle', 'bag_film', 'cup_tray', 'lid_cap', 'carton', 'can', 'foam', 'other_packaging']
Total CLIP prompts: 219
Prompts per class (approx): {'background': 15, 'bottle': 24, 'bag_film': 30, 'cup_tray': 36, 'lid_cap': 24, 'carton': 24, 'can': 24, 'foam': 24, 'other_packaging': 18}


In [32]:
import re

IOSB_SYNONYMS = {
  "bottle": ["a plastic bottle", "a PET bottle", "a water bottle", "a soda bottle"],
  "bag_film": ["a plastic bag", "plastic film", "shrink wrap", "wrapping film", "plastic wrapper"],
  "cup_tray": ["a plastic cup", "a food tray", "a plastic tray", "a takeaway container", "a clamshell container"],
  "lid_cap": ["a bottle cap", "a plastic cap", "a lid", "a container lid"],
  "carton": ["a beverage carton", "a milk carton", "a juice carton", "a liquid carton (Tetra Pak)"],
  "can": ["a metal can", "an aluminum can", "a soda can", "a tin can"],
  "foam": ["foam packaging", "styrofoam", "expanded polystyrene foam", "foam tray"],
  "other_packaging": ["other packaging", "miscellaneous packaging", "unknown packaging item"],
}

IOSB_TEMPLATES = [
  "a waste sorting image containing {}",
  "an industrial waste stream showing {}",
  "a conveyor-belt scene with {}",
  "a waste item: {}",
  "an image of {} in a waste sorting setting",
  "a piece of {} packaging on a conveyor belt",
]

BLIP_BG_PATTERNS = [r"\bconveyor\b", r"\bbelt\b", r"\bempty\b", r"\bbackground\b", r"\bno object\b"]

BLIP_KEYWORDS = {
    "bottle": [r"\bbottle\b", r"\bpet\b"],
    "bag_film": [r"\bbag\b", r"\bfilm\b", r"\bwrap\b", r"\bwrapper\b"],
    "cup_tray": [r"\bcup\b", r"\btray\b", r"\bcontainer\b", r"\bclamshell\b"],
    "lid_cap": [r"\bcap\b", r"\blid\b"],
    "carton": [r"\bcarton\b", r"\btetra\b", r"\btetra pak\b"],
    "can": [r"\bcan\b", r"\baluminum\b", r"\btin\b", r"\bmetal\b"],
    "foam": [r"\bfoam\b", r"\bstyrofoam\b", r"\bpolystyrene\b", r"\beps\b"],
    "other_packaging": [r"\bpackaging\b"],  # fallback bucket
}


@torch.no_grad()
def blip_caption_to_cid_cliptext(caption: str, eva, temp_calib=0.07):
    """
    fallback: EVA-CLIP similarity between caption and (class-synonym-template) prompts.
    returns (cid, conf)
    """
    if caption is None or len(caption.strip()) == 0:
        return 0, 0.0

    # build prompt bank once
    if not hasattr(eva, "_waste_prompt_bank"):
        prompts = []
        p2c = []
        # background prompts
        for ph in ["background", "conveyor belt", "empty belt", "belt surface", "no object"]:
            for t in IOSB_TEMPLATES[:3]:
                prompts.append(t.format(ph))
                p2c.append(0)
        # class prompts
        for cid, cname in enumerate(CAT_NAMES):
            if cname == "background":
                continue
            syns = IOSB_SYNONYMS.get(cname, [cname.replace("_", " ")])
            for s in syns:
                for t in IOSB_TEMPLATES:
                    prompts.append(t.format(s))
                    p2c.append(cid)

        eva._waste_prompts = prompts
        eva._waste_p2c = torch.tensor(p2c, device=device, dtype=torch.long)

        embs = eva.encode_text(prompts)   # depends on your wrapper API
        embs = embs / (embs.norm(dim=-1, keepdim=True) + 1e-8)
        eva._waste_prompt_bank = embs

    txt = eva.encode_text([caption])
    txt = txt / (txt.norm(dim=-1, keepdim=True) + 1e-8)

    sims = (txt @ eva._waste_prompt_bank.T).squeeze(0)  # (P,)
    best_sim = []
    for cid in range(NUM_CLASSES):
        m = (eva._waste_p2c == cid)
        best_sim.append(sims[m].max())
    best_sim = torch.stack(best_sim)  # (C,)

    probs = torch.softmax(best_sim / temp_calib, dim=0)
    cid = int(probs.argmax().item())
    conf = float(probs[cid].item())
    return cid, conf

def blip_caption_to_cid(caption: str, eva, conf_thresh=0.35):
    cid = blip_caption_to_cid_rule(caption)
    if cid is not None:
        return cid, "rule", 1.0
    cid, conf = blip_caption_to_cid_cliptext(caption, eva)
    return cid, "clip_text", conf


In [33]:
def sanitize_label_map(lbl: np.ndarray, num_classes=NUM_CLASSES, ignore=IGNORE):
    """
    Ensures output is only {0..num_classes-1} or ignore.
    If your generator produces 1..C for foreground and 0 unused, this keeps it consistent.
    """
    lbl = lbl.astype(np.int64)

    # common bug: labels are 1..C (foreground) and 0 never used (or background missing)
    # If background is missing but there are values in [1..num_classes], keep as-is.
    # If you want to *force* a shift, uncomment the shift rule below.
    #
    # SHIFT RULE (optional):
    # if lbl.min() >= 1 and lbl.max() <= num_classes:
    #     lbl = lbl - 1

    bad = (lbl != ignore) & ((lbl < 0) | (lbl >= num_classes))
    if bad.any():
        lbl[bad] = ignore
    return lbl.astype(np.uint8)


In [34]:
from pathlib import Path
import json
import numpy as np
from PIL import Image

WASTE = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix")

VAL_IMGS_ROOT = WASTE / "val_imgs"
VAL_GT_ROOT   = WASTE / "val_lbls_form9"   # GT masks

# ---- FIXED ontology (do NOT rely on json ordering) ----
IOSB_CLASSES_9 = [
    "background",
    "bottle",
    "bag_film",
    "cup_tray",
    "lid_cap",
    "carton",
    "can",
    "foam",
    "other_packaging",
]
CAT_NAMES = IOSB_CLASSES_9[:]          # global class list used everywhere
NUM_CLASSES = len(CAT_NAMES)
IGNORE = 255

print("NUM_CLASSES:", NUM_CLASSES)
print("CAT_NAMES:", CAT_NAMES)



NUM_CLASSES: 9
CAT_NAMES: ['background', 'bottle', 'bag_film', 'cup_tray', 'lid_cap', 'carton', 'can', 'foam', 'other_packaging']


In [35]:
# ---- EVA-CLIP base ----
clip_model, clip_preprocess, clip_tokenizer = load_eva_clip(
    device=device,
    model_name="EVA02-L-14",
    pretrained="merged2b_s4b_b131k",
    to_float32=False,
)

# ---- load your WASTE finetuned checkpoint ----
ckpt_path = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_iosb_packform9_FULL.pt"
import torch
ckpt = torch.load(ckpt_path, map_location="cpu")

# find state dict
if isinstance(ckpt, dict) and "model" in ckpt:
    sd = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    sd = ckpt["state_dict"]
elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    sd = ckpt["model_state_dict"]
elif isinstance(ckpt, dict) and any(isinstance(v, torch.Tensor) for v in ckpt.values()):
    sd = ckpt
else:
    raise ValueError(f"Unknown checkpoint format. Keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

missing = clip_model.load_state_dict(sd, strict=False)
print("Loaded. Missing/unexpected:", missing)

# optional: restore logit_scale if saved
if isinstance(ckpt, dict) and "logit_scale_exp" in ckpt:
    with torch.no_grad():
        clip_model.logit_scale.copy_(torch.log(torch.tensor(ckpt["logit_scale_exp"])))
    print("Restored logit_scale from logit_scale_exp.")

eva = EVACLIPWrapper(
    clip_model,
    clip_preprocess,
    clip_tokenizer,
    device=device,
    combine="add",
    embed_dim=1024,
).to(device).eval()

print("Loaded finetuned EVA-CLIP checkpoint:", ckpt_path)
print("EVA wrapper ready.")


Loaded. Missing/unexpected: <All keys matched successfully>
Loaded finetuned EVA-CLIP checkpoint: /home/scs_deal_projects_notapebackup/user/shubhang/thesis/third_party/checkpoints/eva_clip_ft_iosb_packform9_FULL.pt
EVA wrapper ready.


In [36]:
val_img_paths = sorted(VAL_IMGS_ROOT.glob("*.png"))
print("Waste val images:", len(val_img_paths))
print("Example:", val_img_paths[0])

def gt_path_from_img(img_path: Path) -> Path:
    # assumes same filename in GT folder
    return VAL_GT_ROOT / img_path.name

print("GT example:", gt_path_from_img(val_img_paths[0]), "exists:", gt_path_from_img(val_img_paths[0]).exists())


Waste val images: 280
Example: /home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix/val_imgs/record1-fast_broad_conveyor_part1__frame00000_Probe00000.png
GT example: /home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix/val_lbls_form9/record1-fast_broad_conveyor_part1__frame00000_Probe00000.png exists: True


In [37]:
OUT_ROOT = WASTE / "pseudo_lobbe_form9_GTVAL_eval"
OUT_CLIP = OUT_ROOT / "clip_only_v2"
OUT_CLIPBLIP = OUT_ROOT / "clip_plus_blip_v2"

for d in [OUT_CLIP, OUT_CLIPBLIP]:
    (d / "imgs").mkdir(parents=True, exist_ok=True)
    (d / "lbls").mkdir(parents=True, exist_ok=True)
    (d / "meta").mkdir(parents=True, exist_ok=True)

print("Writing to:", OUT_ROOT)


Writing to: /home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix/pseudo_lobbe_form9_GTVAL_eval


In [38]:
import numpy as np
import torch

# ------------ knobs (tune down if needed) ------------
SAM_MAX_SIDE = 640        # 640 is safer than 1024 for vit_h in your current GPU pressure
SAM_GRID_N   = 16         # points per side (16 -> 256 prompts). If still heavy: 12 or 8
SAM_TOPK     = 60         # cap how many masks you return
SAM_MIN_AREA = 800
MIN_MASK_AREA = 800   # match your SAM min area / waste scale (try 500–1500 if needed)


def _resize_np_max_side(np_img: np.ndarray, max_side: int):
    # np_img: HxWx3 uint8
    H, W = np_img.shape[:2]
    s = max(H, W)
    if s <= max_side:
        return np_img, (H, W), 1.0
    scale = max_side / float(s)
    newW = int(round(W * scale))
    newH = int(round(H * scale))
    pil = Image.fromarray(np_img)
    pil = pil.resize((newW, newH), resample=Image.BILINEAR)
    return np.array(pil), (H, W), scale

def _upsample_bool_mask(mask_bool: np.ndarray, target_hw):
    Ht, Wt = target_hw
    m = Image.fromarray(mask_bool.astype(np.uint8) * 255)
    m = m.resize((Wt, Ht), resample=Image.NEAREST)
    return (np.array(m) > 127)

def _grid_points(H, W, n=16):
    # returns Nx2 points in XY, and labels=1
    xs = np.linspace(0.05*W, 0.95*W, n)
    ys = np.linspace(0.05*H, 0.95*H, n)
    xv, yv = np.meshgrid(xs, ys)
    pts = np.stack([xv.reshape(-1), yv.reshape(-1)], axis=1).astype(np.float32)
    lbl = np.ones((pts.shape[0],), dtype=np.int32)
    return pts, lbl

def sam_predict_masks(predictor, np_image: np.ndarray):
    """
    Low-memory SAM mask generation.
    Tries to use SamPredictor.predict() with a grid of point prompts.
    Falls back to predictor.generate() ONLY if no predict() API exists.
    Returns list of HxW uint8 {0,1}.
    """
    if np_image.dtype != np.uint8:
        np_image = np_image.astype(np.uint8)

    # downscale first
    np_small, (H0, W0), _ = _resize_np_max_side(np_image, SAM_MAX_SIDE)
    Hs, Ws = np_small.shape[:2]

    torch.cuda.empty_cache()

    # --- Preferred: SamPredictor API (cheap-ish, no AMG crops) ---
    if hasattr(predictor, "set_image") and hasattr(predictor, "predict"):
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            predictor.set_image(np_small)
            pts, lbls = _grid_points(Hs, Ws, n=SAM_GRID_N)

            # predict masks for ALL points at once (may still be heavy if n too large)
            # if OOM: chunk the points (see chunked version below)
            masks, scores, _ = predictor.predict(
                point_coords=pts,
                point_labels=lbls,
                multimask_output=True,
            )

        # masks: (K, Hs, Ws) or (N*3, Hs, Ws) depending on SAM version
        masks = masks.astype(bool)

        # Score-sort + unique + area filter + topK
        # Flatten if needed
        if masks.ndim == 3:
            cand = [(masks[i], float(scores[i]) if np.ndim(scores) > 0 else 0.0) for i in range(masks.shape[0])]
        else:
            # unexpected, fall back
            cand = [(masks.astype(bool), 0.0)]

        cand.sort(key=lambda x: x[1], reverse=True)

        out = []
        for m, sc in cand:
            if m.sum() < SAM_MIN_AREA:
                continue
            out.append(m)
            if len(out) >= SAM_TOPK:
                break

        # upsample to original
        out_up = [_upsample_bool_mask(m, (H0, W0)).astype(np.uint8) for m in out]
        return out_up

    # --- Fallback: AMG path (your current one) ---
    # This is what OOMs for you; keep as last resort.
    masks_raw = predictor.generate(np_small)
    # Expect your existing _filter_raw_masks exists; otherwise just take segmentations
    try:
        filtered = _filter_raw_masks(masks_raw, max_keep=SAM_TOPK, min_area=SAM_MIN_AREA, iou_thresh=0.8)
        masks_small = [np.array(m["segmentation_bool"], dtype=np.uint8) for m in filtered]
    except Exception:
        masks_small = [np.array(m["segmentation"], dtype=np.uint8) for m in masks_raw[:SAM_TOPK]]

    masks_up = [_upsample_bool_mask(m.astype(bool), (H0, W0)).astype(np.uint8) for m in masks_small]
    return masks_up


In [39]:
def sam_on_pil(pil_img: Image.Image):
    np_img = np.array(pil_img, dtype=np.uint8)
    masks_list = sam_predict_masks(sam_predictor, np_img)
    return [m.astype(bool) for m in masks_list]


In [40]:
import torch

# global cache
_TEXT_CACHE = {}

@torch.no_grad()
def get_text_embeds_for_cats(eva, cat_names, templates):
    """
    Returns (text_embeds, text_mean) where text_embeds has rows = len(cat_names)*len(templates).
    Caches per (cat_names, templates).
    """
    key = (tuple(cat_names), tuple(templates))
    if key in _TEXT_CACHE:
        return _TEXT_CACHE[key]

    # build prompts: templates × classes
    prompts = []
    for cname in cat_names:
        for t in templates:
            prompts.append(t.format(cname.replace("_", " ")))

    # encode + normalize
    text = eva.encode_text(prompts)  # (C*T, D)
    text = text / (text.norm(dim=-1, keepdim=True) + 1e-8)
    text_mean = text.mean(dim=0, keepdim=True)
    text_mean = text_mean / (text_mean.norm(dim=-1, keepdim=True) + 1e-8)

    _TEXT_CACHE[key] = (text, text_mean)
    return text, text_mean


In [41]:
import torch
import numpy as np

# ---------------------------------------------------------
# Text embed cache (per ontology + templates)  ✅ REQUIRED
# ---------------------------------------------------------
_TEXT_CACHE = {}



@torch.no_grad()
def get_text_embeds_for_cats(cat_names, templates, *, device, clip_model, clip_tokenizer):
    """
    Keyword-only args after * (so device=... works).
    Builds text embeddings for (cat_names × templates) using the underlying CLIP model.
    Returns:
      text_embeds: (C*T, D) normalized
      text_mean  : (1, D)   normalized
    """
    key = (tuple(cat_names), tuple(templates))
    if key in _TEXT_CACHE:
        return _TEXT_CACHE[key]

    prompts = []
    for cname in cat_names:
        cname_txt = cname.replace("_", " ")
        for t in templates:
            prompts.append(t.format(cname_txt))

    toks = clip_tokenizer(prompts)

    # tokenizer may return tensor OR dict; move to device
    if isinstance(toks, dict):
        toks = {k: v.to(device) for k, v in toks.items()}
    else:
        toks = toks.to(device)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        if hasattr(clip_model, "encode_text"):
            text = clip_model.encode_text(toks)
        elif hasattr(clip_model, "get_text_features"):
            text = clip_model.get_text_features(**toks) if isinstance(toks, dict) else clip_model.get_text_features(toks)
        else:
            raise AttributeError("clip_model has neither encode_text nor get_text_features")

    text = text / (text.norm(dim=-1, keepdim=True) + 1e-8)
    text_mean = text.mean(dim=0, keepdim=True)
    text_mean = text_mean / (text_mean.norm(dim=-1, keepdim=True) + 1e-8)

    _TEXT_CACHE[key] = (text, text_mean)
    return text, text_mean


import numpy as np
import torch

import numpy as np
import torch

@torch.no_grad()
def classify_masks_with_clip(pil_img, masks_bool):
    W, H = pil_img.size
    results = []

    # ✅ correct embeds for current ontology (waste: 9 classes)
    text_embeds, text_mean = get_text_embeds_for_cats(
        CAT_NAMES, DEFAULT_TEMPLATES,
        device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer
    )

    for midx, mask in enumerate(masks_bool):
        bbox = _mask_to_bbox(mask, pad=4)
        if bbox is None:
            continue
        x0, y0, x1, y1 = bbox

        crop = crop_image(pil_img, bbox)
        mask_crop = mask[y0:y1, x0:x1]
        crop_soft = apply_soft_mask_weighting_strong(crop, mask_crop)

        alpha, beta, _ = dynamic_alpha_beta_from_box((x0, y0, x1, y1), W, H)

        logits_by_class, probs, *_ = eva.classify_crop_with_context_residual_topk(
            crop_pil=crop_soft,
            context_pil=pil_img,
            text_embeds=text_embeds,
            text_feat_mean=text_mean,
            alpha=alpha, beta=beta,
            cat_names=CAT_NAMES,
            templates=DEFAULT_TEMPLATES,
            topk_list=(1, 3, 5),
        )

        logits_np = logits_by_class.detach().cpu().numpy()
        probs_np  = probs.detach().cpu().numpy()
        pred_idx  = int(probs_np.argmax())

        results.append({
            "mask_index": midx,
            "bbox": bbox,
            "mask": mask,
            "logits": logits_np,
            "probs": probs_np,
            "max_prob": float(probs_np.max()),
            "pred_idx": pred_idx,
            "pred_label": CAT_NAMES[pred_idx],
        })

    return results


In [42]:
_TEXT_CACHE.clear()
print("cleared text cache")


cleared text cache


In [43]:
te, tm = get_text_embeds_for_cats(CAT_NAMES, DEFAULT_TEMPLATES, device, clip_model, clip_tokenizer)
print("C=", len(CAT_NAMES), "T=", len(DEFAULT_TEMPLATES), "rows=", te.shape[0])
assert te.shape[0] == len(CAT_NAMES) * len(DEFAULT_TEMPLATES)



TypeError: get_text_embeds_for_cats() takes 2 positional arguments but 5 were given

In [ ]:
import torch
import numpy as np

# separate cache for waste prompt bank
_WASTE_BANK = {}

@torch.no_grad()
def _encode_text_list(prompts, *, device, clip_model, clip_tokenizer):
    """
    Encode a list of strings into normalized CLIP text embeddings (N, D).
    Works with tensor or dict tokenizers.
    """
    toks = clip_tokenizer(prompts)
    if isinstance(toks, dict):
        toks = {k: v.to(device) for k, v in toks.items()}
    else:
        toks = toks.to(device)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        if hasattr(clip_model, "encode_text"):
            z = clip_model.encode_text(toks)
        elif hasattr(clip_model, "get_text_features"):
            z = clip_model.get_text_features(**toks) if isinstance(toks, dict) else clip_model.get_text_features(toks)
        else:
            raise AttributeError("clip_model has neither encode_text nor get_text_features")

    z = z / (z.norm(dim=-1, keepdim=True) + 1e-8)
    return z


@torch.no_grad()
def _build_waste_prompt_bank(*, device, clip_model, clip_tokenizer):
    """
    Build a prompt bank for waste ontology:
      - prompts: list[str]
      - p2c    : tensor[int] mapping prompt index -> class id in 0..8
      - bank   : tensor[float16/32] normalized embeddings (P, D)
    Cached in _WASTE_BANK.
    """
    key = (tuple(CAT_NAMES), tuple(IOSB_TEMPLATES))
    if key in _WASTE_BANK:
        return _WASTE_BANK[key]

    prompts = []
    p2c = []

    # background prompts (class 0)
    for ph in ["background", "conveyor belt", "empty belt", "belt surface", "no object"]:
        for t in IOSB_TEMPLATES[:3]:
            prompts.append(t.format(ph))
            p2c.append(0)

    # class prompts (1..8)
    for cid, cname in enumerate(CAT_NAMES):
        if cname == "background":
            continue
        syns = IOSB_SYNONYMS.get(cname, [cname.replace("_", " ")])
        for s in syns:
            for t in IOSB_TEMPLATES:
                prompts.append(t.format(s))
                p2c.append(cid)

    p2c = torch.tensor(p2c, device=device, dtype=torch.long)
    bank = _encode_text_list(prompts, device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)  # (P, D)

    _WASTE_BANK[key] = (prompts, p2c, bank)
    return _WASTE_BANK[key]


@torch.no_grad()
def _blip_caption_to_cid_cliptext(caption: str, *, device, clip_model, clip_tokenizer, temp_calib=0.07):
    """
    Fallback mapping: embed the BLIP caption and retrieve best class by max similarity over prompts per class.
    Returns (cid, conf) where conf is softmax over class max-sims.
    """
    if caption is None or len(caption.strip()) == 0:
        return 0, 0.0

    prompts, p2c, bank = _build_waste_prompt_bank(device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)

    txt = _encode_text_list([caption], device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)  # (1, D)

    sims = (txt @ bank.T).squeeze(0)  # (P,)

    # for each class: take the max similarity over its prompts
    best_sim = []
    for cid in range(NUM_CLASSES):
        m = (p2c == cid)
        if bool(m.any()):
            best_sim.append(sims[m].max())
        else:
            best_sim.append(torch.tensor(-1e9, device=device, dtype=sims.dtype))
    best_sim = torch.stack(best_sim)  # (C,)

    probs = torch.softmax(best_sim / temp_calib, dim=0)
    cid = int(probs.argmax().item())
    conf = float(probs[cid].item())
    return cid, conf


def blip_caption_to_cid(caption: str, *, device, clip_model, clip_tokenizer, temp_calib=0.07):
    """
    Unified BLIP->class mapping (waste 9-class):
      1) keyword rule mapping (fast, deterministic)
      2) CLIP-text retrieval fallback (caption embedding vs prompt bank)
    Returns (cid, source, conf)
    """
    cid = _blip_caption_to_cid_rule(caption)
    if cid is not None:
        return int(cid), "rule", 1.0

    cid, conf = _blip_caption_to_cid_cliptext(
        caption,
        device=device,
        clip_model=clip_model,
        clip_tokenizer=clip_tokenizer,
        temp_calib=temp_calib
    )
    return int(cid), "clip_text", float(conf)


In [ ]:
_WASTE_BANK.clear()
print("cleared waste prompt bank cache")



cleared waste prompt bank cache


In [49]:
import torch
import numpy as np

# separate cache for waste prompt bank
_WASTE_BANK = {}

@torch.no_grad()
def _encode_text_list(prompts, *, device, clip_model, clip_tokenizer):
    """
    Encode a list of strings into normalized CLIP text embeddings (N, D).
    Works with tensor or dict tokenizers.
    """
    toks = clip_tokenizer(prompts)
    if isinstance(toks, dict):
        toks = {k: v.to(device) for k, v in toks.items()}
    else:
        toks = toks.to(device)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        if hasattr(clip_model, "encode_text"):
            z = clip_model.encode_text(toks)
        elif hasattr(clip_model, "get_text_features"):
            z = clip_model.get_text_features(**toks) if isinstance(toks, dict) else clip_model.get_text_features(toks)
        else:
            raise AttributeError("clip_model has neither encode_text nor get_text_features")

    z = z / (z.norm(dim=-1, keepdim=True) + 1e-8)
    return z


@torch.no_grad()
def _build_waste_prompt_bank(*, device, clip_model, clip_tokenizer):
    """
    Build a prompt bank for waste ontology:
      - prompts: list[str]
      - p2c    : tensor[int] mapping prompt index -> class id in 0..8
      - bank   : tensor[float16/32] normalized embeddings (P, D)
    Cached in _WASTE_BANK.
    """
    key = (tuple(CAT_NAMES), tuple(IOSB_TEMPLATES))
    if key in _WASTE_BANK:
        return _WASTE_BANK[key]

    prompts = []
    p2c = []

    # background prompts (class 0)
    for ph in ["background", "conveyor belt", "empty belt", "belt surface", "no object"]:
        for t in IOSB_TEMPLATES[:3]:
            prompts.append(t.format(ph))
            p2c.append(0)

    # class prompts (1..8)
    for cid, cname in enumerate(CAT_NAMES):
        if cname == "background":
            continue
        syns = IOSB_SYNONYMS.get(cname, [cname.replace("_", " ")])
        for s in syns:
            for t in IOSB_TEMPLATES:
                prompts.append(t.format(s))
                p2c.append(cid)

    p2c = torch.tensor(p2c, device=device, dtype=torch.long)
    bank = _encode_text_list(prompts, device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)  # (P, D)

    _WASTE_BANK[key] = (prompts, p2c, bank)
    return _WASTE_BANK[key]


@torch.no_grad()
def _blip_caption_to_cid_cliptext(caption: str, *, device, clip_model, clip_tokenizer, temp_calib=0.07):
    """
    Fallback mapping: embed the BLIP caption and retrieve best class by max similarity over prompts per class.
    Returns (cid, conf) where conf is softmax over class max-sims.
    """
    if caption is None or len(caption.strip()) == 0:
        return 0, 0.0

    prompts, p2c, bank = _build_waste_prompt_bank(device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)

    txt = _encode_text_list([caption], device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)  # (1, D)

    sims = (txt @ bank.T).squeeze(0)  # (P,)

    # for each class: take the max similarity over its prompts
    best_sim = []
    for cid in range(NUM_CLASSES):
        m = (p2c == cid)
        if bool(m.any()):
            best_sim.append(sims[m].max())
        else:
            best_sim.append(torch.tensor(-1e9, device=device, dtype=sims.dtype))
    best_sim = torch.stack(best_sim)  # (C,)

    probs = torch.softmax(best_sim / temp_calib, dim=0)
    cid = int(probs.argmax().item())
    conf = float(probs[cid].item())
    return cid, conf


def blip_caption_to_cid(caption: str, *, device, clip_model, clip_tokenizer, temp_calib=0.07):
    """
    Unified BLIP->class mapping (waste 9-class):
      1) keyword rule mapping (fast, deterministic)
      2) CLIP-text retrieval fallback (caption embedding vs prompt bank)
    Returns (cid, source, conf)
    """
    cid = _blip_caption_to_cid_rule(caption)
    if cid is not None:
        return int(cid), "rule", 1.0

    cid, conf = _blip_caption_to_cid_cliptext(
        caption,
        device=device,
        clip_model=clip_model,
        clip_tokenizer=clip_tokenizer,
        temp_calib=temp_calib
    )
    return int(cid), "clip_text", float(conf)


In [50]:
# ============================================================
# BLIP caption -> waste class mapping (rule + CLIP-text fallback)
# Uses clip_model + clip_tokenizer (NOT eva.encode_text)
# ============================================================

_WASTE_BANK = {}

@torch.no_grad()
def _encode_text_list(prompts, device, clip_model, clip_tokenizer):
    toks = clip_tokenizer(prompts)
    if isinstance(toks, dict):
        toks = {k: v.to(device) for k, v in toks.items()}
    else:
        toks = toks.to(device)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        if hasattr(clip_model, "encode_text"):
            z = clip_model.encode_text(toks)
        elif hasattr(clip_model, "get_text_features"):
            z = clip_model.get_text_features(**toks) if isinstance(toks, dict) else clip_model.get_text_features(toks)
        else:
            raise AttributeError("clip_model has neither encode_text nor get_text_features")

    z = z / (z.norm(dim=-1, keepdim=True) + 1e-8)
    return z

@torch.no_grad()
def _build_waste_prompt_bank(device, clip_model, clip_tokenizer):
    key = (tuple(CAT_NAMES), tuple(IOSB_TEMPLATES))
    if key in _WASTE_BANK:
        return _WASTE_BANK[key]

    prompts = []
    p2c = []

    # background prompts
    for ph in ["background", "conveyor belt", "empty belt", "belt surface", "no object"]:
        for t in IOSB_TEMPLATES[:3]:
            prompts.append(t.format(ph))
            p2c.append(0)

    # classes 1..8
    for cid, cname in enumerate(CAT_NAMES):
        if cname == "background":
            continue
        syns = IOSB_SYNONYMS.get(cname, [cname.replace("_", " ")])
        for s in syns:
            for t in IOSB_TEMPLATES:
                prompts.append(t.format(s))
                p2c.append(cid)

    p2c = torch.tensor(p2c, device=device, dtype=torch.long)
    bank = _encode_text_list(prompts, device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)

    _WASTE_BANK[key] = (p2c, bank)
    return _WASTE_BANK[key]

@torch.no_grad()
def _blip_caption_to_cid_cliptext(caption, device, clip_model, clip_tokenizer, temp_calib=0.07):
    if caption is None or len(str(caption).strip()) == 0:
        return 0, 0.0

    p2c, bank = _build_waste_prompt_bank(device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)

    txt = _encode_text_list([str(caption)], device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer)  # (1,D)
    sims = (txt @ bank.T).squeeze(0)  # (P,)

    best_sim = []
    for cid in range(NUM_CLASSES):
        m = (p2c == cid)
        best_sim.append(sims[m].max() if bool(m.any()) else torch.tensor(-1e9, device=device, dtype=sims.dtype))
    best_sim = torch.stack(best_sim)  # (C,)

    probs = torch.softmax(best_sim / temp_calib, dim=0)
    cid = int(probs.argmax().item())
    conf = float(probs[cid].item())
    return cid, conf

def blip_caption_to_cid(caption, *args, **kwargs):
    """
    Keyword-safe wrapper to avoid notebook signature mismatches.
    Returns (cid, source, conf)
    """
    device = kwargs.get("device", None)
    clip_model = kwargs.get("clip_model", None)
    clip_tokenizer = kwargs.get("clip_tokenizer", None)
    temp_calib = kwargs.get("temp_calib", 0.07)

    # allow positional fallback
    if device is None and len(args) >= 1:
        device = args[0]
    if clip_model is None and len(args) >= 2:
        clip_model = args[1]
    if clip_tokenizer is None and len(args) >= 3:
        clip_tokenizer = args[2]

    # 1) rules
    cid = _blip_caption_to_cid_rule(caption)
    if cid is not None:
        return int(cid), "rule", 1.0

    # 2) clip-text fallback
    if device is None or clip_model is None or clip_tokenizer is None:
        raise TypeError("blip_caption_to_cid requires device, clip_model, clip_tokenizer")

    cid, conf = _blip_caption_to_cid_cliptext(
        caption, device=device, clip_model=clip_model, clip_tokenizer=clip_tokenizer, temp_calib=temp_calib
    )
    return int(cid), "clip_text", float(conf)


In [51]:
# ============================================================
# Waste (IOSB/Lobbe) ontology + shared globals for pseudo-labeling
# ============================================================
import re
import numpy as np
import torch
from PIL import Image

# ---- Ontology: 9 classes, background first ----
IOSB_CLASSES_9 = [
    "background",
    "bottle",
    "bag_film",
    "cup_tray",
    "lid_cap",
    "carton",
    "can",
    "foam",
    "other_packaging",
]
CAT_NAMES   = IOSB_CLASSES_9[:]     # length 9
NUM_CLASSES = 9
IGNORE_LABEL = 255

# ---------------------------
# Knobs shared by both functions
# ---------------------------
MAX_MASKS_PER_IMAGE  = 15
CLIP_KEEP_PCAL       = 0.85
CLIP_KEEP_MARGIN     = 0.18

# BLIP routing (how much BLIP gets to try)
MAX_BLIP_PER_IMAGE          = 6     # rescue (from non-confident CLIP)
BLIP_OVERRIDE_MAX_PER_IMAGE = 3     # override (from CLIP-kept but ambiguous)

# BLIP accept / scoring (interpreted as "confidence" now, not cosine sim)
BLIP_MIN_SIM_ACCEPT   = 0.35        # rescue accept threshold (caption->class confidence)
BLIP_SCORE_BOOST      = 0.10        # small boost so BLIP can win overlaps sometimes

# BLIP override behavior
BLIP_OVERRIDE_SIM_MIN = 0.50        # require stronger BLIP confidence to override
BLIP_OVERRIDE_BONUS   = 0.20        # extra bonus only when overriding CLIP
BLIP_OVERRIDE_REQUIRE_TOP2 = False  # if True: only override if BLIP == CLIP top2[1]

# "ambiguous CLIP-kept" definition (controls how often override is attempted)
CLIP_OVERRIDE_MARGIN_SLACK = 0.06
CLIP_OVERRIDE_ENTROPY_MIN  = 1.60

# ============================================================
# Label sanitizer (required)
# ============================================================
def sanitize_label_map(lbl: np.ndarray, num_classes=NUM_CLASSES, ignore=IGNORE_LABEL) -> np.ndarray:
    """
    Force label ids into [0..num_classes-1] U {ignore}.
    """
    lbl = lbl.astype(np.int64)
    bad = (lbl != ignore) & ((lbl < 0) | (lbl >= num_classes))
    if bad.any():
        lbl[bad] = ignore
    return lbl.astype(np.uint8)

# ============================================================
# BLIP caption -> waste class mapping (rule + EVA-CLIP text fallback)
# Requires: eva (EVACLIPWrapper), device, and eva.encode_text([...]) support.
# ============================================================
IOSB_SYNONYMS = {
  "bottle": ["a plastic bottle", "a PET bottle", "a water bottle", "a soda bottle"],
  "bag_film": ["a plastic bag", "plastic film", "shrink wrap", "wrapping film", "plastic wrapper"],
  "cup_tray": ["a plastic cup", "a food tray", "a plastic tray", "a takeaway container", "a clamshell container"],
  "lid_cap": ["a bottle cap", "a plastic cap", "a lid", "a container lid"],
  "carton": ["a beverage carton", "a milk carton", "a juice carton", "a liquid carton (Tetra Pak)"],
  "can": ["a metal can", "an aluminum can", "a soda can", "a tin can"],
  "foam": ["foam packaging", "styrofoam", "expanded polystyrene foam", "foam tray"],
  "other_packaging": ["other packaging", "miscellaneous packaging", "unknown packaging item"],
}
IOSB_TEMPLATES = [
  "a waste sorting image containing {}",
  "an industrial waste stream showing {}",
  "a conveyor-belt scene with {}",
  "a waste item: {}",
  "an image of {} in a waste sorting setting",
  "a piece of {} packaging on a conveyor belt",
]

BLIP_BG_PATTERNS = [r"\bconveyor\b", r"\bbelt\b", r"\bempty\b", r"\bbackground\b", r"\bno object\b"]
BLIP_KEYWORDS = {
    "bottle": [r"\bbottle\b", r"\bpet\b"],
    "bag_film": [r"\bbag\b", r"\bfilm\b", r"\bwrap\b", r"\bwrapper\b"],
    "cup_tray": [r"\bcup\b", r"\btray\b", r"\bcontainer\b", r"\bclamshell\b"],
    "lid_cap": [r"\bcap\b", r"\blid\b"],
    "carton": [r"\bcarton\b", r"\btetra\b", r"\btetra pak\b"],
    "can": [r"\bcan\b", r"\baluminum\b", r"\btin\b", r"\bmetal\b"],
    "foam": [r"\bfoam\b", r"\bstyrofoam\b", r"\bpolystyrene\b", r"\beps\b"],
    "other_packaging": [r"\bpackaging\b", r"\bplastic\b"],  # fallback bucket
}

def _blip_caption_to_cid_rule(caption: str):
    if caption is None:
        return None
    s = caption.lower()
    if any(re.search(p, s) for p in BLIP_BG_PATTERNS):
        return 0
    hits = []
    for cname, pats in BLIP_KEYWORDS.items():
        if any(re.search(p, s) for p in pats):
            hits.append(cname)
    if len(hits) == 1:
        return CAT_NAMES.index(hits[0])
    return None







# ============================================================
# PSEUDO-LABEL FUNCTIONS (WASTE, 9 classes)
# These assume your existing pipeline provides:
#   ensure_eva_image_on, sam_on_pil, prune_sam_masks_with_bg,
#   classify_masks_with_clip, compute_uncertainty_from_logits,
#   build_semantic_label_map, crop_image, apply_soft_mask_weighting,
#   should_trigger_blip, should_accept_blip, blip_caption
# ============================================================

@torch.no_grad()
def make_pseudo_clip_only_v2(pil_img: Image.Image, temp_calib=0.07):
    """
    CLIP-only pseudo-label generation using waste ontology (9 classes).
    Outputs label map in {0..8, 255}.
    """
    ensure_eva_image_on(device)

    W, H = pil_img.size
    masks_bool = sam_on_pil(pil_img)
    masks_keep, hard_ignore, info = prune_sam_masks_with_bg(pil_img, masks_bool)

    clip_results = classify_masks_with_clip(pil_img, masks_keep)
    logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

    kept = []
    for r in clip_results:
        top1, top2, margin_cos, pcal_top1, entropy = compute_uncertainty_from_logits(
            r["logits"], logit_scale_exp=logit_scale_exp, temp_calib=temp_calib
        )

        cid = int(top1)
        if not (0 <= cid < NUM_CLASSES):
            continue

        if (pcal_top1 >= CLIP_KEEP_PCAL) and (margin_cos >= CLIP_KEEP_MARGIN):
            kept.append({
                "mask": r["mask"],
                "class_id": cid,
                "score": float(pcal_top1),
                "bbox": r["bbox"],
            })

    kept = sorted(kept, key=lambda x: x["score"], reverse=True)[:MAX_MASKS_PER_IMAGE]
    lbl = build_semantic_label_map(H, W, kept, ignore_label=IGNORE_LABEL, hard_ignore_masks=hard_ignore)
    lbl = sanitize_label_map(lbl, num_classes=NUM_CLASSES, ignore=IGNORE_LABEL)

    meta = {
        "mode": "clip_only_v2",
        "n_sam_raw": len(masks_bool),
        "n_sam_keep": len(masks_keep),
        "n_hard_ignore": len(hard_ignore),
        **info,
        "n_kept": len(kept),
        "temp_calib": temp_calib,
        "CLIP_KEEP_PCAL": CLIP_KEEP_PCAL,
        "CLIP_KEEP_MARGIN": CLIP_KEEP_MARGIN,
        "MAX_MASKS_PER_IMAGE": MAX_MASKS_PER_IMAGE,
        "NUM_CLASSES": NUM_CLASSES,
        "CAT_NAMES": CAT_NAMES,
    }
    return {"label": lbl, "meta": meta}


@torch.no_grad()
def make_pseudo_clip_plus_blip_v2(pil_img: Image.Image, temp_calib=0.07):
    """
    CLIP+BLIP pseudo-label generation using waste ontology (9 classes).
    BLIP mapping is forced into the same 9 classes.
    Outputs label map in {0..8, 255}.
    """
    ensure_eva_image_on(device)
    W, H = pil_img.size

    # 1) SAM + pruning
    masks_bool = sam_on_pil(pil_img)
    masks_keep, hard_ignore, info = prune_sam_masks_with_bg(pil_img, masks_bool)

    # 2) CLIP logits for kept masks
    clip_results = classify_masks_with_clip(pil_img, masks_keep)
    logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

    clip_instances_by_mask = {}
    blip_rescue_candidates = []
    blip_override_candidates = []

    def _make_blip_crop(bbox, m_bool):
        x0,y0,x1,y1 = bbox
        mask_crop_bool = m_bool[y0:y1, x0:x1].astype(bool)

        if "make_blip_focus_crop" in globals():
            try:
                return make_blip_focus_crop(pil_img, bbox, m_bool, bg_mode="gray", blur_bg=True)
            except Exception:
                pass

        crop = crop_image(pil_img, bbox)
        return apply_soft_mask_weighting(crop, mask_crop_bool)

    # 3) Decide CLIP-kept vs BLIP routing
    for r in clip_results:
        midx = int(r.get("mask_index", -1))
        bbox = r.get("bbox", None)
        if bbox is None:
            continue

        m = r["mask"].astype(bool)
        logits = r["logits"]

        top1, top2, margin_cos, pcal_top1, entropy = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=temp_calib
        )

        cid1 = int(top1)
        cid2 = int(top2)
        if not (0 <= cid1 < NUM_CLASSES):  # skip invalid
            continue
        if not (0 <= cid2 < NUM_CLASSES):
            cid2 = cid1

        label1 = CAT_NAMES[cid1]
        label2 = CAT_NAMES[cid2]

        frac = _bbox_area_frac(bbox, H, W)
        confident = (pcal_top1 >= CLIP_KEEP_PCAL) and (margin_cos >= CLIP_KEEP_MARGIN)

        if confident:
            clip_instances_by_mask[midx] = {
                "mask_index": midx,
                "mask": m,
                "class_id": cid1,
                "score": float(pcal_top1),
                "bbox": bbox,
                "src": "clip",
                "pcal": float(pcal_top1),
                "margin": float(margin_cos),
                "entropy": float(entropy),
                "top2": (label1, label2),
            }

            ambiguous_kept = (margin_cos < (CLIP_KEEP_MARGIN + CLIP_OVERRIDE_MARGIN_SLACK)) or (entropy > CLIP_OVERRIDE_ENTROPY_MIN)
            if ambiguous_kept and (len(blip_override_candidates) < BLIP_OVERRIDE_MAX_PER_IMAGE * 6):
                blip_override_candidates.append({
                    "mask_index": midx,
                    "mask": m,
                    "bbox": bbox,
                    "crop_blip": _make_blip_crop(bbox, m),
                    "clip_label": label1,
                    "clip_top2": (label1, label2),
                    "margin_cos": float(margin_cos),
                    "pcal_top1": float(pcal_top1),
                    "entropy": float(entropy),
                    "bbox_frac": float(frac),
                    "mode": "override",
                })
        else:
            if should_trigger_blip(label1, label2, margin_cos, pcal_top1, entropy, frac):
                blip_rescue_candidates.append({
                    "mask_index": midx,
                    "mask": m,
                    "bbox": bbox,
                    "crop_blip": _make_blip_crop(bbox, m),
                    "clip_top2": (label1, label2),
                    "margin_cos": float(margin_cos),
                    "pcal_top1": float(pcal_top1),
                    "entropy": float(entropy),
                    "bbox_frac": float(frac),
                    "mode": "rescue",
                })

    # 4) Limit BLIP work (pick most uncertain first)
    blip_rescue_candidates = sorted(blip_rescue_candidates, key=lambda e: (e["margin_cos"], -e["entropy"]))[:MAX_BLIP_PER_IMAGE]
    blip_override_candidates = sorted(blip_override_candidates, key=lambda e: (e["margin_cos"], -e["entropy"]))[:BLIP_OVERRIDE_MAX_PER_IMAGE]
    blip_candidates = blip_rescue_candidates + blip_override_candidates

    # 5) Run BLIP captions
    crops = [e["crop_blip"] for e in blip_candidates]
    captions = blip_caption(crops, batch_size=4, max_new_tokens=15) if crops else []

    blip_instances = []
    blip_rejected = []
    overrides_done = 0
    rescue_done = 0

    for e, cap in zip(blip_candidates, captions):

        mapped_idx, blip_src, mapped_conf = blip_caption_to_cid(
        cap,
        device=device,
        clip_model=clip_model,
        clip_tokenizer=clip_tokenizer,
        temp_calib=temp_calib
    )
        mapped_idx = int(mapped_idx)
        mapped_idx = mapped_idx if (0 <= mapped_idx < NUM_CLASSES) else 0

        mapped_label = CAT_NAMES[mapped_idx]
        mapped_sim   = float(mapped_conf)   # treat as confidence
  # confidence proxy

        if e["mode"] == "rescue":
            ok = (
                (mapped_sim >= BLIP_MIN_SIM_ACCEPT) and
                should_accept_blip(mapped_label, mapped_sim, e["clip_top2"][0], e["clip_top2"][1])
            )
            if ok:
                score = min(1.0, mapped_sim + BLIP_SCORE_BOOST)
                blip_instances.append({
                    "mask_index": e["mask_index"],
                    "mask": e["mask"],
                    "class_id": mapped_idx,
                    "score": float(score),
                    "bbox": e["bbox"],
                    "src": "blip_rescue",
                    "caption": cap,
                    "mapped_label": mapped_label,
                    "mapped_sim": mapped_sim,
                    "clip_top2": e["clip_top2"],
                    "blip_src": blip_src,
                })
                rescue_done += 1
            else:
                blip_rejected.append({
                    "mode": "rescue",
                    "caption": cap,
                    "mapped": mapped_label,
                    "conf": mapped_sim,
                    "clip_top2": e["clip_top2"],
                    "blip_src": blip_src,
                })
        else:
            clip_label = e["clip_label"]
            clip_top2  = e["clip_top2"]

            override_ok = (mapped_sim >= BLIP_OVERRIDE_SIM_MIN) and (mapped_label != clip_label)
            if BLIP_OVERRIDE_REQUIRE_TOP2:
                override_ok = override_ok and (mapped_label == clip_top2[1])

            if override_ok:
                clip_instances_by_mask.pop(e["mask_index"], None)
                score = min(1.0, mapped_sim + BLIP_SCORE_BOOST + BLIP_OVERRIDE_BONUS)
                blip_instances.append({
                    "mask_index": e["mask_index"],
                    "mask": e["mask"],
                    "class_id": mapped_idx,
                    "score": float(score),
                    "bbox": e["bbox"],
                    "src": "blip_override",
                    "caption": cap,
                    "mapped_label": mapped_label,
                    "mapped_sim": mapped_sim,
                    "clip_prev": clip_label,
                    "clip_top2": clip_top2,
                    "blip_src": blip_src,
                })
                overrides_done += 1
            else:
                blip_rejected.append({
                    "mode": "override",
                    "caption": cap,
                    "mapped": mapped_label,
                    "conf": mapped_sim,
                    "clip_label": clip_label,
                    "clip_top2": clip_top2,
                    "blip_src": blip_src,
                })

    # 6) Merge + cap final masks
    clip_instances = list(clip_instances_by_mask.values())
    merged = clip_instances + blip_instances
    merged = sorted(merged, key=lambda x: x["score"], reverse=True)[:MAX_MASKS_PER_IMAGE]

    # 7) Build label map + sanitize
    lbl = build_semantic_label_map(H, W, merged, ignore_label=IGNORE_LABEL, hard_ignore_masks=hard_ignore)
    lbl = sanitize_label_map(lbl, num_classes=NUM_CLASSES, ignore=IGNORE_LABEL)

    meta = {
        "mode": "clip_plus_blip_v2",
        "n_sam_raw": len(masks_bool),
        "n_sam_keep": len(masks_keep),
        "n_hard_ignore": len(hard_ignore),
        **info,

        "n_clip_results": len(clip_results),
        "n_clip_kept": len(clip_instances),

        "n_blip_rescue_candidates": len(blip_rescue_candidates),
        "n_blip_override_candidates": len(blip_override_candidates),
        "n_blip_candidates": len(blip_candidates),

        "n_blip_accepted_total": len(blip_instances),
        "n_blip_rescues_done": rescue_done,
        "n_overrides_done": overrides_done,

        "n_final_merged": len(merged),

        "temp_calib": temp_calib,

        # snapshot knobs for reproducibility
        "MAX_MASKS_PER_IMAGE": MAX_MASKS_PER_IMAGE,
        "CLIP_KEEP_PCAL": CLIP_KEEP_PCAL,
        "CLIP_KEEP_MARGIN": CLIP_KEEP_MARGIN,
        "MAX_BLIP_PER_IMAGE": MAX_BLIP_PER_IMAGE,
        "BLIP_OVERRIDE_MAX_PER_IMAGE": BLIP_OVERRIDE_MAX_PER_IMAGE,
        "BLIP_MIN_SIM_ACCEPT": BLIP_MIN_SIM_ACCEPT,
        "BLIP_SCORE_BOOST": BLIP_SCORE_BOOST,
        "BLIP_OVERRIDE_SIM_MIN": BLIP_OVERRIDE_SIM_MIN,
        "BLIP_OVERRIDE_BONUS": BLIP_OVERRIDE_BONUS,
        "BLIP_OVERRIDE_REQUIRE_TOP2": BLIP_OVERRIDE_REQUIRE_TOP2,
        "CLIP_OVERRIDE_MARGIN_SLACK": CLIP_OVERRIDE_MARGIN_SLACK,
        "CLIP_OVERRIDE_ENTROPY_MIN": CLIP_OVERRIDE_ENTROPY_MIN,

        "NUM_CLASSES": NUM_CLASSES,
        "CAT_NAMES": CAT_NAMES,

        "blip_rejected_preview": blip_rejected[:5],
    }

    return {"label": lbl, "meta": meta}


In [53]:
from tqdm import tqdm

# Important: your pseudo functions must use the global CAT_NAMES / NUM_CLASSES
# If they were hard-coded to 19 classes, update them to refer to CAT_NAMES length.

ensure_eva_image_on(device)

for img_path in tqdm(val_img_paths, desc="CLIP-only v2 (waste val)"):
    out = make_pseudo_clip_only_v2(pil_from_path(img_path), temp_calib=0.07)
    out["label"] = sanitize_label_map(out["label"])   # <-- ADD THIS
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIP, img_path, out["label"], meta)

for img_path in tqdm(val_img_paths, desc="CLIP+BLIP v2 (waste val)"):
    out = make_pseudo_clip_plus_blip_v2(pil_from_path(img_path), temp_calib=0.07)
    out["label"] = sanitize_label_map(out["label"])   # <-- ADD THIS
    meta = out["meta"]; meta["image_path"] = str(img_path); meta["cat_names"] = CAT_NAMES
    save_pseudo(OUT_CLIPBLIP, img_path, out["label"], meta)


print("Saved pseudo labels to:", OUT_ROOT)


CLIP+BLIP v2 (waste val): 100%|██████████| 280/280 [38:51<00:00,  8.33s/it]

Saved pseudo labels to: /home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix/pseudo_lobbe_form9_GTVAL_eval


In [54]:
import numpy as np
from PIL import Image

def detect_ignore_value_from_gt(gt_dir: Path, sample_n=5):
    ps = sorted(Path(gt_dir).glob("*.png"))[:sample_n]
    for p in ps:
        y = np.array(Image.open(p))
        if (y == 255).any():
            return 255
    return None  # no explicit ignore

IGNORE = detect_ignore_value_from_gt(VAL_GT_ROOT)
print("Detected IGNORE:", IGNORE)

def _sanitize_pred(pr: np.ndarray, num_classes: int, ignore_value):
    pr = pr.astype(np.int64)
    if ignore_value is not None:
        bad = (pr != ignore_value) & ((pr < 0) | (pr >= num_classes))
        if bad.any():
            pr[bad] = ignore_value
    else:
        # no ignore present -> clamp invalid to background (0)
        pr = np.clip(pr, 0, num_classes - 1)
    return pr

def confusion_matrix_np(pred: np.ndarray, target: np.ndarray, num_classes: int, ignore_value):
    if ignore_value is None:
        k = (target >= 0) & (target < num_classes) & (pred >= 0) & (pred < num_classes)
    else:
        k = (target != ignore_value) & (pred != ignore_value) & \
            (target >= 0) & (target < num_classes) & (pred >= 0) & (pred < num_classes)

    if k.sum() == 0:
        return np.zeros((num_classes, num_classes), dtype=np.int64)

    x = target[k].astype(np.int64) * num_classes + pred[k].astype(np.int64)
    binc = np.bincount(x, minlength=num_classes * num_classes)
    return binc.reshape(num_classes, num_classes)

def metrics_from_cm_np(cm: np.ndarray):
    eps = 1e-8
    diag = np.diag(cm).astype(np.float64)
    sum_row = cm.sum(axis=1).astype(np.float64)
    sum_col = cm.sum(axis=0).astype(np.float64)
    union = sum_row + sum_col - diag
    iou  = (diag + eps) / (union + eps)
    dice = (2*diag + eps) / (sum_row + sum_col + eps)
    return {
        "miou": float(np.nanmean(iou)),
        "mdice": float(np.nanmean(dice)),
        "pixel_accuracy": float((diag.sum() + eps) / (cm.sum() + eps)),
    }

def eval_pseudo_vs_gt(pred_lbl_dir: Path, img_paths):
    pred_lbl_dir = Path(pred_lbl_dir)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    n_used = 0
    n_missing = 0
    valid_gt_total = 0
    labeled_on_valid_gt = 0
    overlap_total = 0
    correct_on_overlap = 0

    for img_path in img_paths:
        pred_path = pred_lbl_dir / img_path.name
        gt_path   = gt_path_from_img(img_path)

        if not pred_path.exists():
            n_missing += 1
            continue

        pr = np.array(Image.open(pred_path))
        gt = np.array(Image.open(gt_path))

        gt = gt.astype(np.int64)
            # if GT has invalid ids, clamp to ignore (prevents trash metrics)
        bad_gt = (gt != IGNORE) & ((gt < 0) | (gt >= NUM_CLASSES))
        if bad_gt.any():
            gt[bad_gt] = IGNORE


        # resize pred to GT shape if needed (NEAREST!)
        if pr.shape != gt.shape:
            pr = np.array(Image.fromarray(pr.astype(np.uint8)).resize((gt.shape[1], gt.shape[0]), resample=Image.NEAREST))

        pr = _sanitize_pred(pr, NUM_CLASSES, IGNORE)
        gt = gt.astype(np.int64)

        # valid GT mask
        if IGNORE is None:
            valid_gt = (gt >= 0) & (gt < NUM_CLASSES)
        else:
            valid_gt = (gt != IGNORE) & (gt >= 0) & (gt < NUM_CLASSES)

        # labeled pixels in pred (not ignore if ignore exists; else all are labeled)
        labeled = np.ones_like(valid_gt, dtype=bool) if IGNORE is None else (pr != IGNORE)
        overlap = valid_gt & labeled

        valid_gt_total      += int(valid_gt.sum())
        labeled_on_valid_gt += int(overlap.sum())

        if overlap.any():
            correct_on_overlap += int((pr[overlap] == gt[overlap]).sum())
            overlap_total      += int(overlap.sum())

        cm += confusion_matrix_np(pr, gt, NUM_CLASSES, IGNORE)
        n_used += 1

    m = metrics_from_cm_np(cm)
    coverage = labeled_on_valid_gt / max(valid_gt_total, 1)
    pixacc_overlap = correct_on_overlap / max(overlap_total, 1)

    print(f"Used {n_used}/{len(img_paths)} preds | missing: {n_missing}")
    print(f"coverage_on_valid_gt: {coverage:.4f} | pixAcc_on_labeled_pixels: {pixacc_overlap:.4f}")
    print(f"mIoU: {m['miou']:.4f} | mDice: {m['mdice']:.4f} | pixAcc(cm): {m['pixel_accuracy']:.4f}")

    m.update({
        "images_used": int(n_used),
        "missing_preds": int(n_missing),
        "coverage_on_valid_gt": float(coverage),
        "pixel_acc_on_labeled_pixels": float(pixacc_overlap),
    })
    return m


Detected IGNORE: None


In [55]:
m_clip_waste = eval_pseudo_vs_gt(OUT_CLIP / "lbls", val_img_paths)
m_blip_waste = eval_pseudo_vs_gt(OUT_CLIPBLIP / "lbls", val_img_paths)

print("\nΔ (BLIP - CLIP):")
for k in ["miou", "mdice", "pixel_accuracy", "coverage_on_valid_gt", "pixel_acc_on_labeled_pixels"]:
    print(k, ":", m_blip_waste[k] - m_clip_waste[k])


Used 280/280 preds | missing: 0
coverage_on_valid_gt: 1.0000 | pixAcc_on_labeled_pixels: 0.0631
mIoU: 0.1665 | mDice: 0.2557 | pixAcc(cm): 0.0631
Used 280/280 preds | missing: 0
coverage_on_valid_gt: 1.0000 | pixAcc_on_labeled_pixels: 0.0631
mIoU: 0.1413 | mDice: 0.2362 | pixAcc(cm): 0.0631

Δ (BLIP - CLIP):
miou : -0.0252656994740989
mdice : -0.019507105819030424
pixel_accuracy : 1.9005579607822565e-05
coverage_on_valid_gt : 0.0
pixel_acc_on_labeled_pixels : 1.9005579607822565e-05


In [56]:
import torch

margins = []
pcals = []
logit_scale_exp = float(eva.model.logit_scale.exp().detach().cpu().item())

for img_path in val_img_paths[:5]:
    pil_img = pil_from_path(img_path)
    masks = select_top_sam_masks(sam_on_pil(pil_img), top_k=25, min_area=800)
    clip_results = classify_masks_with_clip(pil_img, masks)

    for r in clip_results:
        logits = r["logits"]
        if torch.is_tensor(logits):
            logits = logits.detach().cpu().numpy()

        top1, top2, margin_cos, pcal_top1, ent = compute_uncertainty_from_logits(
            logits, logit_scale_exp=logit_scale_exp, temp_calib=0.07
        )
        margins.append(margin_cos)
        pcals.append(pcal_top1)

import numpy as np
print("margin_cos: min/med/max", np.min(margins), np.median(margins), np.max(margins))
print("pcal_top1:  min/med/max", np.min(pcals), np.median(pcals), np.max(pcals))


margin_cos: min/med/max 0.0006670355796813965 0.10698512941598892 0.27152788639068604
pcal_top1:  min/med/max 0.29768556356430054 0.7054396569728851 0.9507200717926025


In [60]:
from pathlib import Path
import numpy as np
from PIL import Image

NUM_CLASSES = 9
IGNORE = 255

def sanitize_ids(arr: np.ndarray, num_classes=NUM_CLASSES, ignore=IGNORE) -> np.ndarray:
    arr = arr.astype(np.int64)
    bad = (arr != ignore) & ((arr < 0) | (arr >= num_classes))
    if bad.any():
        arr[bad] = ignore
    return arr

def fast_confmat(gt, pr, k):
    return np.bincount(k * gt + pr, minlength=k * k).reshape(k, k)

def detect_ignore_value(mask_dir: str | Path, sample_n=8) -> int | None:
    ps = sorted(Path(mask_dir).glob("*.png"))[:sample_n]
    for p in ps:
        y = np.array(Image.open(p))
        if (y == 255).any():
            return 255
    return None

def eval_pseudo_vs_gt_common_waste(pseudo_lbl_dir, gt_lbl_dir, num_classes=NUM_CLASSES, ignore=None):
    pseudo_lbl_dir = Path(pseudo_lbl_dir)
    gt_lbl_dir = Path(gt_lbl_dir)

    if ignore is None:
        ignore = detect_ignore_value(gt_lbl_dir)
    print("IGNORE used:", ignore)

    gt_files = {p.name: p for p in gt_lbl_dir.glob("*.png")}
    pr_files = {p.name: p for p in pseudo_lbl_dir.glob("*.png")}
    common = sorted(set(gt_files.keys()) & set(pr_files.keys()))
    print("common files:", len(common))

    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    valid_gt_total = labeled_on_valid_gt = correct_on_overlap = overlap_total = 0

    for fn in common:
        gt = np.array(Image.open(gt_files[fn]), dtype=np.int64)
        pr = np.array(Image.open(pr_files[fn]), dtype=np.int64)

        if pr.shape != gt.shape:
            pr = np.array(
                Image.fromarray(pr.astype(np.uint8)).resize((gt.shape[1], gt.shape[0]), resample=Image.NEAREST),
                dtype=np.int64
            )

        if ignore is None:
            gt = np.clip(gt, 0, num_classes - 1)
            pr = np.clip(pr, 0, num_classes - 1)
            valid_gt = np.ones_like(gt, dtype=bool)
            labeled  = np.ones_like(pr, dtype=bool)
        else:
            gt = sanitize_ids(gt, num_classes=num_classes, ignore=ignore)
            pr = sanitize_ids(pr, num_classes=num_classes, ignore=ignore)
            valid_gt = (gt != ignore)
            labeled  = (pr != ignore)

        overlap = valid_gt & labeled

        valid_gt_total      += int(valid_gt.sum())
        labeled_on_valid_gt += int(overlap.sum())

        if overlap.any():
            gt_o = gt[overlap].astype(np.int64)
            pr_o = pr[overlap].astype(np.int64)

            correct_on_overlap += int((gt_o == pr_o).sum())
            overlap_total      += int(overlap.sum())

            # both guaranteed 0..k-1 here
            cm += fast_confmat(gt_o, pr_o, num_classes)

    coverage = labeled_on_valid_gt / max(valid_gt_total, 1)
    pixacc   = correct_on_overlap / max(overlap_total, 1)

    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    denom = tp + fp + fn
    iou = np.where(denom > 0, tp / denom, np.nan)

    present = np.where(cm.sum(1) > 0)[0]
    miou_present = float(np.nanmean(iou[present])) if len(present) else float("nan")

    return {
        "images_used": len(common),
        "coverage_on_valid_gt": float(coverage),
        "pixel_acc_on_labeled_pixels": float(pixacc),
        "mIoU_on_labeled_pixels_present_classes": miou_present,
        "num_classes": int(num_classes),
        "ignore": ignore,
    }


In [61]:
from pathlib import Path

WASTE = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix")

GT_LBLS_WASTE   = str(WASTE / "val_lbls_form9")
CLIP_LBLS_WASTE = str(OUT_CLIP / "lbls")
BLIP_LBLS_WASTE = str(OUT_CLIPBLIP / "lbls")

print("[PSEUDO vs GT] CLIP-only:", eval_pseudo_vs_gt_common_waste(CLIP_LBLS_WASTE, GT_LBLS_WASTE))
print("[PSEUDO vs GT] CLIP+BLIP:", eval_pseudo_vs_gt_common_waste(BLIP_LBLS_WASTE, GT_LBLS_WASTE))


IGNORE used: None
common files: 280
[PSEUDO vs GT] CLIP-only: {'images_used': 280, 'coverage_on_valid_gt': 1.0, 'pixel_acc_on_labeled_pixels': 0.06311416349240712, 'mIoU_on_labeled_pixels_present_classes': 0.1665219425642636, 'num_classes': 9, 'ignore': None}
IGNORE used: None
common files: 280
[PSEUDO vs GT] CLIP+BLIP: {'images_used': 280, 'coverage_on_valid_gt': 1.0, 'pixel_acc_on_labeled_pixels': 0.06313316907201494, 'mIoU_on_labeled_pixels_present_classes': 0.14125624309016466, 'num_classes': 9, 'ignore': None}


In [59]:
import numpy as np
from PIL import Image
from pathlib import Path

ps = sorted(Path(GT_LBLS_WASTE).glob("*.png"))[:3]
for p in ps:
    y = np.array(Image.open(p))
    u = np.unique(y)
    print(p.name, "min", int(u.min()), "max", int(u.max()),
          "has255", bool((y==255).any()),
          "uniq_head", u[:20])


record1-fast_broad_conveyor_part1__frame00000_Probe00000.png min 0 max 8 has255 False uniq_head [0 8]
record1-fast_broad_conveyor_part1__frame00001_Probe00001.png min 0 max 8 has255 False uniq_head [0 2 7 8]
record1-fast_broad_conveyor_part1__frame00002_Probe00002.png min 0 max 8 has255 False uniq_head [0 1 3 4 7 8]


In [ ]:
import os, json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# -----------------------------
# Config
# -----------------------------
CITY_JSONL = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis/pseudo_runs/cityscapes_gta5_evaclip/raw/val_pseudo.jsonl"
WASTE_JSONL = "/home/scs_deal_projects_notapebackup/user/shubhang/thesis/pseudo_runs/lobbe/pseudo_val.jsonl"

# You must set these to where your GT masks live (trainId PNGs for Cityscapes; your decoded GT PNGs for Lobbe val)
from pathlib import Path

CITY = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/cityscapes")
CS_VAL_GT_ROOT  = CITY / "gtFine" / "val"

WASTE = Path("/home/scs_deal_projects_notapebackup/shared/DATASET/waste_dataset/lobbe_fix")
VAL_GT_ROOT   = WASTE / "val_lbls_form9"

CITY_GT_DIR  = str(CS_VAL_GT_ROOT)
WASTE_GT_DIR = str(VAL_GT_ROOT)         # whatever your GT mask filenames are

CITY_CLASSES = [
    "road","sidewalk","building","wall","fence","pole","traffic light","traffic sign",
    "vegetation","terrain","sky","person","rider","car","truck","bus","train","motorcycle","bicycle"
]
WASTE_CLASSES = ["bottle","bag_film","cup_tray","lid_cap","carton","can","foam","other_packaging"]

IGNORE_LABEL = 255

# Waste: if you ignore background in evaluation, set BACKGROUND_LABEL accordingly.
# If your GT masks already have background as 255 (ignore), set this to None.
WASTE_BACKGROUND_LABEL = 0  # change if needed (common: 0). If unsure, we auto-diagnose below.


# -----------------------------
# Utilities
# -----------------------------
def read_jsonl(path, n_preview=2):
    """Read jsonl and preview keys."""
    recs = []
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
            rec = json.loads(line)
            recs.append(rec)
            if i + 1 >= n_preview:
                break
    print(f"[preview] {path}")
    for i, r in enumerate(recs):
        print(f"  line {i}: keys = {sorted(list(r.keys()))}")
    return recs

def find_first_key(d, candidates):
    for k in candidates:
        if k in d:
            return k
    return None

def city_gt_from_image_path(img_path: str) -> str:
    """
    Input: .../leftImg8bit/val/<city>/<name>_leftImg8bit.png
    Output: <city>/<name>_gtFine_labelTrainIds.png   (relative to CITY_GT_DIR)
    """
    p = Path(img_path)
    city = p.parent.name
    fname = p.name.replace("_leftImg8bit.png", "_gtFine_labelTrainIds.png")
    return str(Path(city) / fname)

def waste_gt_from_image_name(img_name: str) -> str:
    """Default: same basename. Adjust if your GT naming differs."""
    return os.path.basename(img_name)

def load_mask_png(path):
    return np.array(Image.open(path), dtype=np.int64)

# Try pycocotools for COCO RLE decoding
try:
    from pycocotools import mask as coco_mask
    HAVE_COCO = True
except Exception as e:
    HAVE_COCO = False
    print("[warn] pycocotools not available. Install with: pip install pycocotools")
    print("       If your JSONL uses compressed RLE strings, decoding will fail without it.")

def decode_rle(rle_obj):
    """
    Decode COCO-style RLE dict:
    rle_obj = {"counts": ..., "size":[h,w]}
    counts can be list (uncompressed) or string (compressed).
    """
    if not HAVE_COCO:
        raise RuntimeError("pycocotools required for RLE decoding.")
    # coco_mask.decode returns HxWxN or HxW; ensure HxW boolean
    m = coco_mask.decode(rle_obj)
    if m.ndim == 3:
        m = m[:, :, 0]
    return m.astype(bool)

def extract_image_path(rec):
    k = find_first_key(rec, ["image_path","img_path","file_name","filename","image","path"])
    if k is None:
        return None
    return rec[k]

def extract_segments(rec):
    """
    Returns a list of segment dicts or None.
    We try common keys used by pseudo-label pipelines.
    """
    k = find_first_key(rec, ["segments","segmentations","masks","annotations","instances","predictions"])
    if k is None:
        return None
    if isinstance(rec[k], list):
        return rec[k]
    return None

def segment_label(seg):
    """
    Extract class id from a segment entry.
    Tries common keys.
    """
    k = find_first_key(seg, ["category_id","class_id","label_id","trainId","label","cls","category"])
    if k is None:
        return None
    return seg[k]

def segment_score(seg):
    k = find_first_key(seg, ["score","conf","confidence","logit","prob"])
    if k is None:
        return 1.0
    try:
        return float(seg[k])
    except:
        return 1.0

def segment_rle(seg):
    """
    Extract RLE from a segment entry.
    Common keys: "rle", "segmentation" (dict), "mask_rle".
    """
    k = find_first_key(seg, ["rle","mask_rle","segmentation"])
    if k is None:
        return None
    obj = seg[k]
    # Some pipelines wrap segmentation as {"counts":..., "size":[h,w]}
    if isinstance(obj, dict) and "counts" in obj and "size" in obj:
        return obj
    # Some store as {"segmentation": {"counts":..., "size":[h,w]}}
    if isinstance(obj, dict) and "segmentation" in obj and isinstance(obj["segmentation"], dict):
        inner = obj["segmentation"]
        if "counts" in inner and "size" in inner:
            return inner
    return None

def record_has_pseudo_png(rec):
    k = find_first_key(rec, ["pseudo_path","pseudo_mask_path","pseudo_png","mask_path","pred_path"])
    return k

def build_pseudo_map_from_segments(segments, H, W, ignore_label=255):
    """
    Build per-pixel pseudo label map from a list of segments.
    Overlaps resolved by descending score (higher score wins).
    """
    pseudo = np.full((H, W), ignore_label, dtype=np.int64)

    # Sort by score so higher-confidence segments overwrite lower
    segments_sorted = sorted(segments, key=segment_score, reverse=True)

    for seg in segments_sorted:
        cls = segment_label(seg)
        rle = segment_rle(seg)
        if cls is None or rle is None:
            continue
        m = decode_rle(rle)  # boolean mask
        # Safety: match shapes
        if m.shape != pseudo.shape:
            # If rle size differs, resize is NOT correct; skip and inspect.
            raise ValueError(f"Mask shape mismatch: got {m.shape}, expected {(H,W)}")
        pseudo[m] = int(cls)

    return pseudo

def compute_confusion_on_omega(pseudo, gt, class_ids, ignore_gt=255, ignore_pseudo=255, ignore_background=None):
    """
    Compute TP/FP/FN per class restricted to Omega = pseudo-labeled pixels,
    and optionally excluding background GT pixels from denominator.
    """
    # evaluable pixels in GT
    eval_mask = (gt != ignore_gt)
    if ignore_background is not None:
        eval_mask &= (gt != ignore_background)

    omega = (pseudo != ignore_pseudo) & eval_mask

    tp = np.zeros(len(class_ids), dtype=np.int64)
    fp = np.zeros(len(class_ids), dtype=np.int64)
    fn = np.zeros(len(class_ids), dtype=np.int64)

    for i, c in enumerate(class_ids):
        c = int(c)
        pred_c = (pseudo == c)
        gt_c = (gt == c)
        tp[i] = np.sum(pred_c & gt_c & omega)
        fp[i] = np.sum(pred_c & (~gt_c) & omega)
        fn[i] = np.sum((~pred_c) & gt_c & omega)

    # coverage on evaluable pixels
    cov = float(np.sum(omega)) / float(np.sum(eval_mask) + 1e-9)
    return tp, fp, fn, cov

def summarize_iou(tp, fp, fn, class_names):
    denom = tp + fp + fn
    iou = np.where(denom > 0, tp / denom, np.nan)
    df = pd.DataFrame({"class": class_names, "IoU_omega": np.round(iou, 4),
                       "TP": tp, "FP": fp, "FN": fn})
    miou = np.nanmean(iou)
    return df, float(miou)

def run_jsonl_eval(jsonl_path, gt_dir, class_names, gt_name_fn, ignore_gt=255, ignore_pseudo=255, ignore_background=None, max_items=None):
    class_ids = list(range(len(class_names)))

    tp_all = np.zeros(len(class_ids), dtype=np.int64)
    fp_all = np.zeros(len(class_ids), dtype=np.int64)
    fn_all = np.zeros(len(class_ids), dtype=np.int64)
    covs = []

    with open(jsonl_path, "r") as f:
        for idx, line in enumerate(tqdm(f, desc=os.path.basename(jsonl_path))):
            if not line.strip():
                continue
            rec = json.loads(line)

            img_path = extract_image_path(rec)
            if img_path is None:
                # If your JSONL uses an "id", you'll need a mapping table; print to debug
                raise KeyError("Could not find image path key in record. Inspect keys in preview.")

            # Locate GT
            gt_name = gt_name_fn(img_path)
            gt_path = os.path.join(gt_dir, gt_name)
            if not os.path.exists(gt_path):
                raise FileNotFoundError(f"GT not found: {gt_path}")

            gt = load_mask_png(gt_path)
            H, W = gt.shape[:2]

            # Build pseudo map
            pseudo_key = record_has_pseudo_png(rec)
            if pseudo_key is not None:
                pseudo_path = rec[pseudo_key]
                pseudo = load_mask_png(pseudo_path)
                if pseudo.shape != gt.shape:
                    raise ValueError(f"Pseudo/GT shape mismatch: {pseudo.shape} vs {gt.shape}")
            else:
                segments = extract_segments(rec)
                if segments is None:
                    raise KeyError("Could not find segments list or pseudo mask path in record.")
                pseudo = build_pseudo_map_from_segments(segments, H, W, ignore_label=ignore_pseudo)

            tp, fp, fn, cov = compute_confusion_on_omega(
                pseudo=pseudo, gt=gt,
                class_ids=class_ids,
                ignore_gt=ignore_gt,
                ignore_pseudo=ignore_pseudo,
                ignore_background=ignore_background
            )

            tp_all += tp
            fp_all += fp
            fn_all += fn
            covs.append(cov)

            if max_items is not None and (idx + 1) >= max_items:
                break

    df, miou = summarize_iou(tp_all, fp_all, fn_all, class_names)
    cov_mean = float(np.mean(covs)) if covs else 0.0
    return df, miou, cov_mean


# -----------------------------
# Preview keys (so you can adjust extractors if needed)
# -----------------------------
_ = read_jsonl(CITY_JSONL, n_preview=2)
_ = read_jsonl(WASTE_JSONL, n_preview=2)

# -----------------------------
# Optional: auto-diagnose waste background label (from a few GT masks)
# -----------------------------
def guess_background_label(gt_dir, n=10):
    vals = []
    files = [f for f in os.listdir(gt_dir) if f.lower().endswith(".png")]
    for f in files[:n]:
        gt = load_mask_png(os.path.join(gt_dir, f))
        uniq = np.unique(gt)
        vals.append(uniq)
    uniq_all = np.unique(np.concatenate(vals))
    return uniq_all

# print("Waste GT unique labels:", guess_background_label(WASTE_GT_DIR, n=5))


# -----------------------------
# Run evaluation
# -----------------------------
df_city, miou_city, cov_city = run_jsonl_eval(
    jsonl_path=CITY_JSONL,
    gt_dir=CITY_GT_DIR,
    class_names=CITY_CLASSES,
    gt_name_fn= city_gt_from_image_path,
    ignore_gt=IGNORE_LABEL,
    ignore_pseudo=IGNORE_LABEL,
    ignore_background=None
)

df_waste, miou_waste, cov_waste = run_jsonl_eval(
    jsonl_path=WASTE_JSONL,
    gt_dir=WASTE_GT_DIR,
    class_names=WASTE_CLASSES,
    gt_name_fn=waste_gt_from_image_name,
    ignore_gt=IGNORE_LABEL,
    ignore_pseudo=IGNORE_LABEL,
    ignore_background=WASTE_BACKGROUND_LABEL
)

print("Cityscapes pseudo-label quality (Omega): mIoU =", miou_city, "coverage =", cov_city)
display(df_city)

print("Waste pseudo-label quality (Omega): mIoU =", miou_waste, "coverage =", cov_waste)
display(df_waste)

# Export LaTeX tables
print(df_city[["class","IoU_omega"]].to_latex(index=False))
print(df_waste[["class","IoU_omega"]].to_latex(index=False))